# Exact embeddings

This notebook downloads the sentence-embedding model all-MiniLM-L6-v2 and the 2,758 sentences of the STS-B test set at pinned revisions and checks both by SHA-256. It embeds every sentence with an integer version of the model, in which the only floating-point operation is a product of integer matrices small enough to be exact in double precision, and with a plain float32 version. It prints the STS-B Spearman correlation of both and, if the runtime has a GPU and JAX, runs the integer model on the GPU. The last cell prints the SHA-256 of the 2,758 integer vectors with MATCH or DIFFERENT against the hash in the repository, then the SHA-256 of the float32 vectors on this machine.


In [ ]:
# The folder's code and tables, written here and checked by SHA-256.
import hashlib, os
FILES = {}
FILES['exact_embed.py'] = ('e3e7c1e29512d99a55254d994250463c55135b47f91413e1f789dfeed9346d08', '"""Exact sentence embeddings: sentence-transformers/all-MiniLM-L6-v2 in integer arithmetic.\n\nEvery tensor is an integer array with a power-of-two scale. Element-wise work is NumPy int64. The only floating-point\noperation is matrix multiplication, done by the machine\'s double-precision BLAS on integer-valued arrays; every\npartial sum is an integer below 2^53 (bounds.py proves it for the scales below), so the product is exact in any\norder, with or without fused multiply-add, on any number of threads. GELU and exp are integer tables\n(tables/, made by make_tables.py). The output is 384 int16 per sentence at scale 2^-15.\n\nThe same file holds the pinned downloads, a BERT WordPiece tokenizer in plain Python, a plain float32 forward pass\nof the same model (the baseline), an onnxruntime baseline, and the STS-B evaluation.\n\nusage:\n  python exact_embed.py check                      digests, exact hash against expected.sha256, baselines, STS-B\n  python exact_embed.py hash --engine exact|float32|onnx [--batch N] [--threads N]\n"""\nimport gzip\nimport hashlib\nimport json\nimport math\nimport os\nimport struct\nimport sys\nimport unicodedata\nimport urllib.request\n\nimport numpy as np\n\nHERE = os.path.dirname(os.path.abspath(__file__))\nPINS = json.load(open(os.path.join(HERE, "pins.json")))\nCACHE = os.environ.get("EXACT_EMBED_CACHE", os.path.join(HERE, ".cache"))\n\n# ---- scales ---------------------------------------------------------------------------------------------------\nA = 16                        # activations: integer x stands for x * 2^-16\nWB = 16                       # weights: integer w stands for w * 2^-16; biases at 2^-(A + WB)\nCLAMP = 1 << (7 + A)          # every activation is clamped to [-2^7, 2^7] (real units) before a product\nCLAMP_LN = 1 << (9 + A)       # LayerNorm inputs are clamped to [-2^9, 2^9]\nOUT = 15                      # output: int16 at 2^-15\nHEADS, HEAD = 12, 32\nHIDDEN = 384\nEXP_STEP, EXP_ONE, EXP_N = 8, 24, 24 * 256     # exp table: grid step 2^-8, values at 2^-24, last point 24\nGELU_STEP, GELU_LO, GELU_N = 8, 8, 16 * 256    # gelu table: grid step 2^-8 on [-8, 8], values at 2^-16\n\n\n# ---- downloads ------------------------------------------------------------------------------------------------\ndef sha256_file(path):\n    h = hashlib.sha256()\n    with open(path, "rb") as f:\n        for chunk in iter(lambda: f.read(1 << 20), b""):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef fetch(url, path, digest):\n    """Download url to path unless a file with that SHA-256 is already there; check the digest either way."""\n    if not (os.path.exists(path) and sha256_file(path) == digest):\n        os.makedirs(os.path.dirname(path), exist_ok=True)\n        req = urllib.request.Request(url, headers={"User-Agent": "exact-embeddings"})\n        with urllib.request.urlopen(req) as r, open(path + ".part", "wb") as f:\n            while True:\n                chunk = r.read(1 << 20)\n                if not chunk:\n                    break\n                f.write(chunk)\n        os.replace(path + ".part", path)\n    got = sha256_file(path)\n    if got != digest:\n        raise SystemExit("digest mismatch for %s: %s, pinned %s" % (path, got, digest))\n    return path\n\n\ndef model_file(name):\n    m = PINS["model"]\n    url = "https://huggingface.co/%s/resolve/%s/%s" % (m["repo"], m["revision"], name)\n    return fetch(url, os.path.join(CACHE, "model", name), m["files"][name])\n\n\ndef sentences_and_scores():\n    s = PINS["sentences"]\n    url = "https://huggingface.co/datasets/%s/resolve/%s/%s" % (s["repo"], s["revision"], s["file"])\n    path = fetch(url, os.path.join(CACHE, "data", s["file"]), s["sha256"])\n    rows = [json.loads(line) for line in gzip.open(path, "rt", encoding="utf-8")]\n    assert len(rows) == s["pairs"]\n    sentences = []\n    for r in rows:\n        sentences += [r["sentence1"], r["sentence2"]]\n    return sentences, np.array([float(r["score"]) for r in rows])\n\n\n# ---- safetensors, parsed here: 8-byte little-endian header length, JSON header, raw little-endian tensors --------\ndef read_safetensors(path):\n    with open(path, "rb") as f:\n        data = f.read()\n    n = struct.unpack("<Q", data[:8])[0]\n    header = json.loads(data[8:8 + n].decode("utf-8"))\n    header.pop("__metadata__", None)\n    base = 8 + n\n    out = {}\n    for name, t in header.items():\n        if t["dtype"] != "F32":\n            continue\n        b, e = t["data_offsets"]\n        count = int(np.prod(t["shape"])) if t["shape"] else 1\n        assert e - b == 4 * count\n        out[name] = np.frombuffer(data, dtype="<f4", count=count, offset=base + b).reshape(t["shape"])\n    return out\n\n\n# ---- tokenizer: BERT WordPiece, uncased, as in the model\'s tokenizer_config.json --------------------------------\ndef _is_whitespace(c):\n    return c in " \\t\\n\\r" or unicodedata.category(c) == "Zs"\n\n\ndef _is_control(c):\n    return c not in "\\t\\n\\r" and unicodedata.category(c).startswith("C")\n\n\ndef _is_punctuation(c):\n    cp = ord(c)\n    if 33 <= cp <= 47 or 58 <= cp <= 64 or 91 <= cp <= 96 or 123 <= cp <= 126:\n        return True\n    return unicodedata.category(c).startswith("P")\n\n\ndef _is_cjk(cp):\n    return (0x4E00 <= cp <= 0x9FFF or 0x3400 <= cp <= 0x4DBF or 0x20000 <= cp <= 0x2A6DF or 0x2A700 <= cp <= 0x2B73F\n            or 0x2B740 <= cp <= 0x2B81F or 0x2B820 <= cp <= 0x2CEAF or 0xF900 <= cp <= 0xFAFF\n            or 0x2F800 <= cp <= 0x2FA1F)\n\n\nclass Tokenizer:\n    def __init__(self, vocab_path, max_len=256):\n        self.vocab = {}\n        with open(vocab_path, encoding="utf-8") as f:\n            for i, line in enumerate(f):\n                self.vocab[line.rstrip("\\n")] = i\n        self.max_len = max_len\n        self.cls, self.sep, self.unk = self.vocab["[CLS]"], self.vocab["[SEP]"], self.vocab["[UNK]"]\n\n    def basic(self, text):\n        out = []\n        for c in text:\n            cp = ord(c)\n            if cp == 0 or cp == 0xFFFD or _is_control(c):\n                continue\n            if _is_whitespace(c):\n                out.append(" ")\n            elif _is_cjk(cp):\n                out.append(" %s " % c)\n            else:\n                out.append(c)\n        words = []\n        for w in "".join(out).split():\n            w = w.lower()\n            w = "".join(c for c in unicodedata.normalize("NFD", w) if unicodedata.category(c) != "Mn")\n            cur = ""\n            for c in w:\n                if _is_punctuation(c):\n                    if cur:\n                        words.append(cur)\n                        cur = ""\n                    words.append(c)\n                else:\n                    cur += c\n            if cur:\n                words.append(cur)\n        return words\n\n    def wordpiece(self, word):\n        if len(word) > 100:\n            return [self.unk]\n        pieces, start = [], 0\n        while start < len(word):\n            end, found = len(word), None\n            while start < end:\n                sub = word[start:end] if start == 0 else "##" + word[start:end]\n                if sub in self.vocab:\n                    found = self.vocab[sub]\n                    break\n                end -= 1\n            if found is None:\n                return [self.unk]\n            pieces.append(found)\n            start = end\n        return pieces\n\n    def encode(self, text):\n        ids = []\n        for w in self.basic(text):\n            ids += self.wordpiece(w)\n        return [self.cls] + ids[: self.max_len - 2] + [self.sep]\n\n\ndef batches(tok, sentences, size):\n    for i in range(0, len(sentences), size):\n        enc = [tok.encode(s) for s in sentences[i:i + size]]\n        L = max(len(e) for e in enc)\n        ids = np.zeros((len(enc), L), dtype=np.int64)\n        mask = np.zeros((len(enc), L), dtype=np.int64)\n        for j, e in enumerate(enc):\n            ids[j, :len(e)] = e\n            mask[j, :len(e)] = 1\n        yield ids, mask\n\n\n# ---- integer arithmetic ---------------------------------------------------------------------------------------\ndef rshift_round(v, k):\n    """floor(v / 2^k + 1/2): round to nearest, halves up."""\n    return (v + (1 << (k - 1))) >> k\n\n\ndef isqrt(n):\n    """Exact floor(sqrt(n)) for int64 0 <= n < 2^62: float64 sqrt, then integer correction steps.\n    The result s is asserted by its bracket s^2 <= n < (s + 1)^2, the form a proof circuit checks."""\n    assert (n >= 0).all() and (n < (1 << 62)).all()\n    s = np.sqrt(n.astype(np.float64)).astype(np.int64)\n    for _ in range(4):\n        s = np.where(s * s > n, s - 1, s)\n        s = np.where((s + 1) * (s + 1) <= n, s + 1, s)\n    assert ((s * s <= n) & ((s + 1) * (s + 1) > n)).all()\n    return s\n\n\ndef fdiv(num, den):\n    """floor(num / den) for den > 0, asserted by its bracket q den <= num < (q + 1) den."""\n    den = np.asarray(den, dtype=np.int64)\n    assert (den > 0).all()\n    q = num // den\n    assert ((q * den <= num) & (num < (q + 1) * den)).all()\n    return q\n\n\ndef load_table(name):\n    with open(os.path.join(HERE, "tables", name)) as f:\n        return np.array([int(line) for line in f if not line.startswith("#")], dtype=np.int64)\n\n\ndef round_pow2(a, k):\n    """round(a * 2^k), a float32, to the nearest integer (ties to even). Exact: the scaling is exact in float64."""\n    return np.rint(a.astype(np.float64) * 2.0 ** k).astype(np.int64)\n\n\ndef round_pow2_over_sqrt32(a, k):\n    """round(a * 2^k / sqrt(32)) = round(a * 2^(k-3) * sqrt(2)), in exact integer arithmetic (no ties occur)."""\n    m = k - 3\n    flat = a.astype(np.float64).ravel().tolist()\n    out = np.zeros(len(flat), dtype=np.int64)\n    for i, v in enumerate(flat):\n        if v == 0.0:\n            continue\n        num, den = v.as_integer_ratio()\n        p, q = (abs(num) << m, den) if m >= 0 else (abs(num), den << -m)\n        n = (math.isqrt((8 * p * p) // (q * q)) + 1) // 2          # floor(p sqrt2 / q + 1/2)\n        out[i] = n if num > 0 else -n\n    return out.reshape(a.shape)\n\n\nclass Exact:\n    def __init__(self, params, keep_int=False):\n        P = params\n        self.word = round_pow2(P["embeddings.word_embeddings.weight"], A)\n        self.pos = round_pow2(P["embeddings.position_embeddings.weight"], A)\n        self.typ = round_pow2(P["embeddings.token_type_embeddings.weight"], A)\n        self.emb_ln = (round_pow2(P["embeddings.LayerNorm.weight"], A), round_pow2(P["embeddings.LayerNorm.bias"], A))\n        self.layers = []\n        self.int_weights = {}\n        for i in range(6):\n            p = "encoder.layer.%d." % i\n            lay = {}\n            for key, name in (("q", "attention.self.query"), ("k", "attention.self.key"),\n                              ("v", "attention.self.value"), ("o", "attention.output.dense"),\n                              ("i", "intermediate.dense"), ("o2", "output.dense")):\n                w, b = P[p + name + ".weight"], P[p + name + ".bias"]\n                if key == "q":                       # 1/sqrt(32) folded into the query before rounding\n                    wi, bi = round_pow2_over_sqrt32(w, WB), round_pow2_over_sqrt32(b, A + WB)\n                else:\n                    wi, bi = round_pow2(w, WB), round_pow2(b, A + WB)\n                if keep_int:\n                    self.int_weights[p + name + ".weight"] = wi\n                lay[key] = (np.ascontiguousarray(wi.T).astype(np.float64), bi)\n            lay["ln1"] = (round_pow2(P[p + "attention.output.LayerNorm.weight"], A),\n                          round_pow2(P[p + "attention.output.LayerNorm.bias"], A))\n            lay["ln2"] = (round_pow2(P[p + "output.LayerNorm.weight"], A), round_pow2(P[p + "output.LayerNorm.bias"], A))\n            self.layers.append(lay)\n        self.E = load_table("exp.txt")\n        self.G = load_table("gelu.txt")\n        assert len(self.E) == EXP_N + 1 and len(self.G) == GELU_N + 1\n        self.clamped = {}\n        self.peak = {}\n\n    def clamp(self, x, c, where):\n        """Clamp to [-c, c], counting the values that were outside and keeping the largest magnitude seen."""\n        a = np.abs(x)\n        n = int((a > c).sum())\n        if n:\n            self.clamped[where] = self.clamped.get(where, 0) + n\n        self.peak[where] = max(self.peak.get(where, 0), int(a.max()) if a.size else 0)\n        return np.clip(x, -c, c)\n\n    def linear(self, x, wb, where):\n        w, b = wb\n        y = (self.clamp(x, CLAMP, where).astype(np.float64) @ w).astype(np.int64) + b      # exact, see bounds.py\n        return rshift_round(y, WB)\n\n    def layernorm(self, x, gb):\n        g, b = gb\n        x = self.clamp(x, CLAMP_LN, "layernorm input")\n        mu = fdiv(x.sum(-1, keepdims=True), HIDDEN)\n        d = x - mu\n        var = fdiv((d * d).sum(-1, keepdims=True), HIDDEN)           # at 2^-32\n        std = isqrt(var + 1)                                          # at 2^-16; the + 1 is eps = 2^-32\n        y = fdiv(d << A, std)\n        return rshift_round(y * g, A) + b\n\n    def exp(self, t):\n        """E interpolated: t >= 0 at 2^-16; result at 2^-24."""\n        k = np.minimum(t >> EXP_STEP, EXP_N)\n        r = t & ((1 << EXP_STEP) - 1)\n        e0, e1 = self.E[k], self.E[np.minimum(k + 1, EXP_N)]\n        return (e0 * ((1 << EXP_STEP) - r) + e1 * r) >> EXP_STEP\n\n    def gelu(self, x):\n        """G interpolated on [-8, 8); x outside: 0 below, x above. x and result at 2^-16."""\n        u = x + (GELU_LO << A)\n        inside = (u >= 0) & (u < (GELU_N << GELU_STEP))\n        uc = np.clip(u, 0, (GELU_N << GELU_STEP) - 1)\n        k, r = uc >> GELU_STEP, uc & ((1 << GELU_STEP) - 1)\n        g = (self.G[k] * ((1 << GELU_STEP) - r) + self.G[k + 1] * r) >> GELU_STEP\n        return np.where(inside, g, np.where(u < 0, 0, x))\n\n    def attention(self, q, k, v, valid):\n        qc = self.clamp(q, CLAMP, "query").astype(np.float64)\n        kc = self.clamp(k, CLAMP, "key").astype(np.float64)\n        vc = self.clamp(v, CLAMP, "value").astype(np.float64)\n        s = (qc @ kc.swapaxes(-1, -2)).astype(np.int64)              # scores at 2^-32, |s| <= 32 * 2^46\n        key = valid[:, None, None, :]\n        m = np.where(key, s, -(1 << 62)).max(-1, keepdims=True)\n        t = np.maximum(m - s, 0) >> A                                 # m - s at 2^-16, floor\n        e = np.where(key, self.exp(t), 0)                             # at 2^-24\n        p = fdiv(e << A, e.sum(-1, keepdims=True))                    # at 2^-16, each row sums to at most 2^16\n        ctx = (p.astype(np.float64) @ vc).astype(np.int64)            # at 2^-32, |ctx| <= 2^16 * 2^23\n        return rshift_round(ctx, A)\n\n    def forward(self, ids, mask):\n        B, L = ids.shape\n        valid = mask.astype(bool)\n        x = self.word[ids] + self.pos[:L][None] + self.typ[0][None, None]\n        x = self.layernorm(x, self.emb_ln)\n        for n, lay in enumerate(self.layers):\n            h = x.reshape(B * L, HIDDEN)\n            split = lambda y: y.reshape(B, L, HEADS, HEAD).transpose(0, 2, 1, 3)\n            q = split(self.linear(h, lay["q"], "query input"))\n            k = split(self.linear(h, lay["k"], "key input"))\n            v = split(self.linear(h, lay["v"], "value input"))\n            ctx = self.attention(q, k, v, valid).transpose(0, 2, 1, 3).reshape(B * L, HIDDEN)\n            a = self.linear(ctx, lay["o"], "attention output input")\n            x = self.layernorm((h + a).reshape(B, L, HIDDEN), lay["ln1"])\n            h = x.reshape(B * L, HIDDEN)\n            f = self.gelu(self.linear(h, lay["i"], "intermediate input"))\n            o = self.linear(f, lay["o2"], "output input")\n            x = self.layernorm((h + o).reshape(B, L, HIDDEN), lay["ln2"])\n        return self.pool(x, valid)\n\n    def pool(self, x, valid):\n        x = self.clamp(x, CLAMP, "pooling input")\n        s = (x * valid[..., None]).sum(1)\n        pooled = fdiv(s, valid.sum(1, keepdims=True).astype(np.int64))  # mean over the attention mask, floor, 2^-16\n        norm = isqrt((pooled * pooled).sum(-1, keepdims=True))        # at 2^-16\n        out = fdiv(pooled << OUT, np.maximum(norm, 1))                # floor, 2^-15\n        return np.clip(out, -(1 << OUT), (1 << OUT) - 1).astype(np.int16)\n\n\n# ---- the float32 baseline: a plain forward pass of the same model -------------------------------------------------\ndef _erf32(x):\n    """erf by Abramowitz and Stegun 7.1.26 in float64 (error below 1.5e-7), returned as float32."""\n    z = np.abs(x.astype(np.float64))\n    t = 1.0 / (1.0 + 0.3275911 * z)\n    y = 1.0 - (((((1.061405429 * t - 1.453152027) * t) + 1.421413741) * t - 0.284496736) * t + 0.254829592) * t \\\n        * np.exp(-z * z)\n    return (np.sign(x) * y).astype(np.float32)\n\n\nclass Float32:\n    def __init__(self, params):\n        self.P = {k: np.ascontiguousarray(v) for k, v in params.items()}\n        self.T = {k: np.ascontiguousarray(v.T) for k, v in params.items() if k.endswith(".weight") and v.ndim == 2}\n\n    def ln(self, x, p):\n        mu = x.mean(-1, keepdims=True)\n        var = ((x - mu) ** 2).mean(-1, keepdims=True)\n        return (x - mu) / np.sqrt(var + np.float32(1e-12)) * self.P[p + ".weight"] + self.P[p + ".bias"]\n\n    def lin(self, x, p):\n        return x @ self.T[p + ".weight"] + self.P[p + ".bias"]\n\n    def forward(self, ids, mask):\n        B, L = ids.shape\n        P = self.P\n        x = P["embeddings.word_embeddings.weight"][ids] + P["embeddings.position_embeddings.weight"][:L][None] \\\n            + P["embeddings.token_type_embeddings.weight"][0][None, None]\n        x = self.ln(x, "embeddings.LayerNorm")\n        neg = ((1.0 - mask.astype(np.float32)) * np.finfo(np.float32).min)[:, None, None, :]\n        for i in range(6):\n            p = "encoder.layer.%d." % i\n            split = lambda y: y.reshape(B, L, HEADS, HEAD).transpose(0, 2, 1, 3)\n            q, k, v = (split(self.lin(x, p + "attention.self." + n)) for n in ("query", "key", "value"))\n            s = (q @ k.swapaxes(-1, -2)) / np.float32(math.sqrt(HEAD)) + neg\n            s = np.exp(s - s.max(-1, keepdims=True))\n            ctx = ((s / s.sum(-1, keepdims=True)) @ v).transpose(0, 2, 1, 3).reshape(B, L, HIDDEN)\n            x = self.ln(x + self.lin(ctx, p + "attention.output.dense"), p + "attention.output.LayerNorm")\n            h = self.lin(x, p + "intermediate.dense")\n            h = h * np.float32(0.5) * (np.float32(1.0) + _erf32(h / np.float32(math.sqrt(2.0))))\n            x = self.ln(x + self.lin(h, p + "output.dense"), p + "output.LayerNorm")\n        return pool_float(x, mask)\n\n\ndef pool_float(x, mask):\n    m = mask.astype(np.float32)[..., None]\n    pooled = (x * m).sum(1) / np.maximum(m.sum(1), np.float32(1e-9))\n    return (pooled / np.maximum(np.linalg.norm(pooled, axis=-1, keepdims=True), np.float32(1e-12))).astype(np.float32)\n\n\nclass Onnx:\n    def __init__(self, threads=0):\n        import onnxruntime as ort\n        so = ort.SessionOptions()\n        if threads:\n            so.intra_op_num_threads = threads\n            so.inter_op_num_threads = 1\n        self.s = ort.InferenceSession(model_file("onnx/model.onnx"), so, providers=["CPUExecutionProvider"])\n        self.names = [i.name for i in self.s.get_inputs()]\n\n    def forward(self, ids, mask):\n        feed = {"input_ids": ids, "attention_mask": mask, "token_type_ids": np.zeros_like(ids)}\n        out = self.s.run(None, {k: v for k, v in feed.items() if k in self.names})[0]\n        return pool_float(out.astype(np.float32), mask)\n\n\n# ---- running, hashing, STS-B ----------------------------------------------------------------------------------\ndef load_params():\n    return read_safetensors(model_file("model.safetensors"))\n\n\ndef engine(name, threads=0):\n    if name == "exact":\n        return Exact(load_params())\n    if name == "float32":\n        return Float32(load_params())\n    if name == "onnx":\n        return Onnx(threads)\n    raise SystemExit("unknown engine " + name)\n\n\ndef embed(eng, tok, sentences, batch):\n    return np.concatenate([eng.forward(ids, mask) for ids, mask in batches(tok, sentences, batch)])\n\n\ndef digest(vectors):\n    dt = "<i2" if vectors.dtype == np.int16 else "<f4"\n    return hashlib.sha256(np.ascontiguousarray(vectors).astype(dt).tobytes()).hexdigest()\n\n\ndef rankdata(a):\n    _, inv, counts = np.unique(a, return_inverse=True, return_counts=True)\n    end = np.cumsum(counts)\n    return ((end - counts + end - 1) / 2.0 + 1.0)[inv]\n\n\ndef spearman(x, y):\n    rx, ry = rankdata(x), rankdata(y)\n    rx, ry = rx - rx.mean(), ry - ry.mean()\n    return float((rx * ry).sum() / math.sqrt((rx * rx).sum() * (ry * ry).sum()))\n\n\ndef pair_cosines(v):\n    a, b = v[0::2], v[1::2]\n    if v.dtype == np.int16:\n        return (a.astype(np.int64) * b.astype(np.int64)).sum(1)      # the integer dot product, cosine * 2^30\n    return (a.astype(np.float64) * b.astype(np.float64)).sum(1)\n\n\ndef tokenizer():\n    return Tokenizer(model_file("vocab.txt"), max_len=256)\n\n\ndef expected():\n    return open(os.path.join(HERE, "expected.sha256")).read().split()[0]\n\n\ndef main(argv):\n    if len(argv) > 1 and argv[1] == "hash":\n        eng_name = argv[argv.index("--engine") + 1] if "--engine" in argv else "exact"\n        batch = int(argv[argv.index("--batch") + 1]) if "--batch" in argv else 32\n        threads = int(argv[argv.index("--threads") + 1]) if "--threads" in argv else 0\n        sentences, _ = sentences_and_scores()\n        v = embed(engine(eng_name, threads), tokenizer(), sentences, batch)\n        print("%s %s batch=%d" % (digest(v), eng_name, batch))\n        if "--save" in argv:\n            np.save(argv[argv.index("--save") + 1], v)\n        return 0\n    if len(argv) > 1 and argv[1] == "check":\n        sentences, gold = sentences_and_scores()\n        tok = tokenizer()\n        ex = Exact(load_params())\n        v = embed(ex, tok, sentences, 32)\n        h = digest(v)\n        ok = h == expected()\n        print("exact    %s  %s" % (h, "MATCH" if ok else "DIFFERENT"))\n        print("         clamped values: %d" % sum(ex.clamped.values()))\n        f = embed(Float32(load_params()), tok, sentences, 32)\n        print("float32  %s" % digest(f))\n        try:\n            o = embed(Onnx(), tok, sentences, 32)\n            print("onnx     %s" % digest(o))\n        except ImportError:\n            o = None\n            print("onnx     onnxruntime is not installed here")\n        print("STS-B test, %d pairs, Spearman of cosine against the gold scores:" % len(gold))\n        print("  exact    %.4f" % spearman(pair_cosines(v), gold))\n        print("  float32  %.4f" % spearman(pair_cosines(f), gold))\n        if o is not None:\n            print("  onnx     %.4f" % spearman(pair_cosines(o), gold))\n        e = v.astype(np.float64) / 2.0 ** OUT\n        c = (e * f).sum(1) / np.linalg.norm(e, axis=1) / np.linalg.norm(f.astype(np.float64), axis=1)\n        print("cosine between the float32 and the exact vector of each sentence: min %.6f, mean %.6f" % (c.min(), c.mean()))\n        return 0 if ok else 1\n    print(__doc__)\n    return 2\n\n\nif __name__ == "__main__":\n    sys.exit(main(sys.argv))\n')
FILES['jax_runner.py'] = ('52b1546a253f458998582e9ca878ee164b159c838b550c5afb42c44508b57aa8', '"""The exact model of exact_embed.py, executed by JAX in int64, on CPU or GPU.\n\nThe same integer semantics, written again with jax.numpy: every tensor is int64 (jax_enable_x64), every product is an\ninteger einsum, the tables are gathers, the square root is digit by digit, divisions are floor divisions. There is\nno floating point on this path at all. bounds.json gives the int64 headroom of every product and sum. The weights are\nthe integers exact_embed.py rounds from the published float32 values. The result must have the SHA-256 in\nexpected.sha256, the same as the NumPy executor\'s.\n\nusage: python jax_runner.py [--device cpu|gpu] [--batch N]\n"""\nimport os\nimport sys\nimport time\n\nimport numpy as np\n\nHERE = os.path.dirname(os.path.abspath(__file__))\nsys.path.insert(0, HERE)\nimport exact_embed as X  # noqa: E402\n\nimport jax  # noqa: E402\n\njax.config.update("jax_enable_x64", True)\nimport jax.numpy as jnp  # noqa: E402\n\nI64 = jnp.int64\n\n\ndef rshift_round(v, k):\n    return (v + (1 << (k - 1))) >> k\n\n\ndef isqrt(n):\n    """floor(sqrt(n)) for 0 <= n < 2^62, digit by digit in integers."""\n    rem, res, bit = n, jnp.zeros_like(n), 1 << 60\n    for _ in range(31):\n        t = res + bit\n        take = rem >= t\n        rem = jnp.where(take, rem - t, rem)\n        res = jnp.where(take, (res >> 1) + bit, res >> 1)\n        bit >>= 2\n    return res\n\n\ndef mm(spec, a, b):\n    return jnp.einsum(spec, a, b, preferred_element_type=I64)\n\n\ndef make_forward():\n    """Returns forward(W, ids, mask) -> int16 [B, 384], jitted. W, the integer weights and tables as int64 arrays,\n    is an argument, not a closure, so the compiled code does not carry the weights as constants."""\n    C, CLN, A, WB = X.CLAMP, X.CLAMP_LN, X.A, X.WB\n\n    def linear(x, w, b):\n        return rshift_round(mm("ni,io->no", jnp.clip(x, -C, C), w) + b, WB)\n\n    def layernorm(x, g, b):\n        x = jnp.clip(x, -CLN, CLN)\n        mu = jnp.floor_divide(x.sum(-1, keepdims=True), X.HIDDEN)\n        d = x - mu\n        var = jnp.floor_divide((d * d).sum(-1, keepdims=True), X.HIDDEN)\n        std = isqrt(var + 1)\n        y = jnp.floor_divide(d << A, std)\n        return rshift_round(y * g, A) + b\n\n    def exp(t, E):\n        k = jnp.minimum(t >> X.EXP_STEP, X.EXP_N)\n        r = t & ((1 << X.EXP_STEP) - 1)\n        e0 = jnp.take(E, k)\n        e1 = jnp.take(E, jnp.minimum(k + 1, X.EXP_N))\n        return (e0 * ((1 << X.EXP_STEP) - r) + e1 * r) >> X.EXP_STEP\n\n    def gelu(x, G):\n        top = X.GELU_N << X.GELU_STEP\n        u = x + (X.GELU_LO << A)\n        uc = jnp.clip(u, 0, top - 1)\n        k, r = uc >> X.GELU_STEP, uc & ((1 << X.GELU_STEP) - 1)\n        g = (jnp.take(G, k) * ((1 << X.GELU_STEP) - r) + jnp.take(G, k + 1) * r) >> X.GELU_STEP\n        return jnp.where((u >= 0) & (u < top), g, jnp.where(u < 0, 0, x))\n\n    def attention(q, k, v, valid, E):\n        q, k, v = (jnp.clip(t, -C, C) for t in (q, k, v))\n        s = mm("bhqd,bhkd->bhqk", q, k)\n        key = valid[:, None, None, :]\n        m = jnp.where(key, s, -(1 << 62)).max(-1, keepdims=True)\n        t = jnp.maximum(m - s, 0) >> A\n        e = jnp.where(key, exp(t, E), 0)\n        p = jnp.floor_divide(e << A, e.sum(-1, keepdims=True))\n        return rshift_round(mm("bhqk,bhkd->bhqd", p, v), A)\n\n    def forward(W, ids, mask):\n        B, L = ids.shape\n        valid = mask.astype(bool)\n        x = jnp.take(W["word"], ids, axis=0) + W["pos"][:L][None] + W["typ"][0][None, None]\n        x = layernorm(x, W["emb_g"], W["emb_b"])\n        split = lambda y: y.reshape(B, L, X.HEADS, X.HEAD).transpose(0, 2, 1, 3)\n        for lay in W["layers"]:\n            h = x.reshape(B * L, X.HIDDEN)\n            q, k, v = (split(linear(h, *lay[n])) for n in ("q", "k", "v"))\n            ctx = attention(q, k, v, valid, W["E"]).transpose(0, 2, 1, 3).reshape(B * L, X.HIDDEN)\n            x = layernorm((h + linear(ctx, *lay["o"])).reshape(B, L, X.HIDDEN), *lay["ln1"])\n            h = x.reshape(B * L, X.HIDDEN)\n            o = linear(gelu(linear(h, *lay["i"]), W["G"]), *lay["o2"])\n            x = layernorm((h + o).reshape(B, L, X.HIDDEN), *lay["ln2"])\n        x = jnp.clip(x, -C, C)\n        pooled = jnp.floor_divide((x * valid[..., None]).sum(1), valid.sum(1, keepdims=True).astype(I64))\n        norm = isqrt((pooled * pooled).sum(-1, keepdims=True))\n        out = jnp.floor_divide(pooled << X.OUT, jnp.maximum(norm, 1))\n        return jnp.clip(out, -(1 << X.OUT), (1 << X.OUT) - 1).astype(jnp.int16)\n\n    return jax.jit(forward)\n\n\ndef weights(ex, device):\n    put = lambda a: jax.device_put(np.asarray(a, dtype=np.int64), device)\n    W = {"word": put(ex.word), "pos": put(ex.pos), "typ": put(ex.typ), "emb_g": put(ex.emb_ln[0]),\n         "emb_b": put(ex.emb_ln[1]), "E": put(ex.E), "G": put(ex.G), "layers": []}\n    for lay in ex.layers:\n        d = {}\n        for n in ("q", "k", "v", "o", "i", "o2"):\n            wt, b = lay[n]\n            d[n] = (put(wt.astype(np.int64)), put(b))        # wt holds integers below 2^53: the cast is exact\n        d["ln1"] = (put(lay["ln1"][0]), put(lay["ln1"][1]))\n        d["ln2"] = (put(lay["ln2"][0]), put(lay["ln2"][1]))\n        W["layers"].append(d)\n    return W\n\n\ndef run(device_kind="cpu", batch=32, bucket=8):\n    """Embed the 2,758 sentences. Lengths are padded up to a multiple of `bucket` so that few shapes compile;\n    padding changes no result (padded keys are masked, padded rows are not pooled)."""\n    device = jax.devices(device_kind)[0]\n    sentences, _ = X.sentences_and_scores()\n    tok = X.tokenizer()\n    with jax.default_device(device):\n        W = weights(X.Exact(X.load_params()), device)\n        fwd = make_forward()\n        out = []\n        for ids, mask in X.batches(tok, sentences, batch):\n            L = -(-ids.shape[1] // bucket) * bucket\n            ids = np.pad(ids, ((0, 0), (0, L - ids.shape[1])))\n            mask = np.pad(mask, ((0, 0), (0, L - mask.shape[1])))\n            out.append(np.asarray(fwd(W, jax.device_put(ids, device), jax.device_put(mask, device))))\n    return np.concatenate(out), device\n\n\nif __name__ == "__main__":\n    kind = sys.argv[sys.argv.index("--device") + 1] if "--device" in sys.argv else "cpu"\n    batch = int(sys.argv[sys.argv.index("--batch") + 1]) if "--batch" in sys.argv else 32\n    t0 = time.time()\n    v, dev = run(kind, batch)\n    h = X.digest(v)\n    print("jax int64 on %s (%s), batch %d: %s  %s  (%.0f s)"\n          % (dev.platform, getattr(dev, "device_kind", ""), batch, h, "MATCH" if h == X.expected() else "DIFFERENT",\n             time.time() - t0))\n')
FILES['pins.json'] = ('118920e45f2bc848dc8cc489ef29a98070b7787ee3b714afbc7d271bf7d8392d', '{\n  "model": {\n    "repo": "sentence-transformers/all-MiniLM-L6-v2",\n    "revision": "1110a243fdf4706b3f48f1d95db1a4f5529b4d41",\n    "license": "apache-2.0",\n    "files": {\n      "config.json": "953f9c0d463486b10a6871cc2fd59f223b2c70184f49815e7efbcab5d8908b41",\n      "vocab.txt": "07eced375cec144d27c900241f3e339478dec958f92fddbc551f295c992038a3",\n      "model.safetensors": "53aa51172d142c89d9012cce15ae4d6cc0ca6895895114379cacb4fab128d9db",\n      "onnx/model.onnx": "6fd5d72fe4589f189f8ebc006442dbb529bb7ce38f8082112682524616046452"\n    }\n  },\n  "sentences": {\n    "repo": "mteb/stsbenchmark-sts",\n    "revision": "96943a16ea6a35129e253c659081cb59daf81b30",\n    "file": "test.jsonl.gz",\n    "sha256": "524a75137d413dbfc40c3e84905f733714232647c3848521acb6bc375b834e1c",\n    "pairs": 1379,\n    "order": "for each line of the file in order: sentence1, then sentence2 (2,758 sentences)"\n  }\n}\n')
FILES['expected.sha256'] = ('f45f7adc287956df914b1042b57fba66af5a9c8a149cd2109eccbedbb068b2ec', '031d6912ba5a0824fd2b2562abb592fe5caf8053e0d5a2060cf051a9ab202934  the 2,758 int16 vectors of the exact model, STS-B test sentences in file order (pins.json), little-endian\n')
FILES['tables/exp.txt'] = ('ff0cd05a41e8be53cef25a50a7ee17ed512ec68ec3c43a65a7aea45e27e3d10c', '# exp table\n# E[k] = round(2^24 * exp(-k/256)), k = 0 .. 6144\n# 6145 values, one per line, generated by make_tables.py\n16777216\n16711808\n16646655\n16581756\n16517109\n16452715\n16388572\n16324679\n16261035\n16197640\n16134491\n16071589\n16008931\n15946518\n15884349\n15822422\n15760736\n15699291\n15638085\n15577118\n15516388\n15455896\n15395639\n15335617\n15275829\n15216274\n15156952\n15097860\n15038999\n14980368\n14921965\n14863790\n14805841\n14748119\n14690621\n14633348\n14576298\n14519470\n14462864\n14406479\n14350313\n14294367\n14238638\n14183127\n14127832\n14072753\n14017889\n13963238\n13908801\n13854575\n13800561\n13746758\n13693165\n13639780\n13586604\n13533634\n13480872\n13428315\n13375963\n13323815\n13271870\n13220128\n13168588\n13117248\n13066109\n13015169\n12964428\n12913884\n12863538\n12813388\n12763433\n12713673\n12664107\n12614734\n12565554\n12516566\n12467768\n12419161\n12370743\n12322514\n12274473\n12226620\n12178953\n12131471\n12084175\n12037064\n11990136\n11943390\n11896828\n11850446\n11804246\n11758225\n11712384\n11666722\n11621238\n11575931\n11530801\n11485846\n11441067\n11396463\n11352032\n11307775\n11263690\n11219777\n11176035\n11132464\n11089063\n11045830\n11002767\n10959871\n10917143\n10874581\n10832185\n10789954\n10747888\n10705986\n10664247\n10622671\n10581257\n10540005\n10498913\n10457982\n10417210\n10376597\n10336143\n10295846\n10255706\n10215723\n10175896\n10136224\n10096706\n10057343\n10018133\n9979076\n9940172\n9901418\n9862816\n9824365\n9786063\n9747911\n9709908\n9672052\n9634345\n9596784\n9559370\n9522101\n9484978\n9447999\n9411165\n9374475\n9337927\n9301522\n9265259\n9229137\n9193156\n9157315\n9121614\n9086052\n9050629\n9015344\n8980197\n8945186\n8910312\n8875574\n8840972\n8806504\n8772171\n8737971\n8703905\n8669972\n8636171\n8602501\n8568963\n8535556\n8502279\n8469132\n8436114\n8403225\n8370464\n8337830\n8305324\n8272945\n8240692\n8208564\n8176562\n8144685\n8112932\n8081302\n8049796\n8018413\n7987152\n7956013\n7924996\n7894099\n7863323\n7832667\n7802130\n7771712\n7741413\n7711232\n7681169\n7651223\n7621394\n7591681\n7562084\n7532602\n7503235\n7473983\n7444844\n7415820\n7386908\n7358109\n7329423\n7300848\n7272385\n7244032\n7215791\n7187659\n7159637\n7131724\n7103920\n7076225\n7048637\n7021157\n6993784\n6966518\n6939358\n6912304\n6885356\n6858512\n6831773\n6805139\n6778608\n6752181\n6725856\n6699635\n6673515\n6647498\n6621582\n6595767\n6570052\n6544438\n6518924\n6493509\n6468193\n6442976\n6417857\n6392836\n6367913\n6343087\n6318357\n6293724\n6269188\n6244746\n6220400\n6196149\n6171993\n6147931\n6123962\n6100087\n6076305\n6052616\n6029019\n6005514\n5982101\n5958779\n5935548\n5912407\n5889357\n5866396\n5843525\n5820744\n5798051\n5775446\n5752930\n5730501\n5708160\n5685906\n5663739\n5641658\n5619663\n5597754\n5575931\n5554192\n5532539\n5510969\n5489484\n5468083\n5446765\n5425530\n5404378\n5383308\n5362320\n5341415\n5320590\n5299847\n5279185\n5258604\n5238102\n5217681\n5197339\n5177077\n5156893\n5136788\n5116762\n5096813\n5076943\n5057150\n5037434\n5017795\n4998232\n4978746\n4959336\n4940001\n4920742\n4901558\n4882448\n4863413\n4844453\n4825566\n4806753\n4788013\n4769346\n4750752\n4732231\n4713782\n4695405\n4677099\n4658865\n4640701\n4622609\n4604587\n4586636\n4568754\n4550942\n4533200\n4515526\n4497922\n4480386\n4462919\n4445520\n4428188\n4410924\n4393728\n4376598\n4359536\n4342539\n4325609\n4308745\n4291947\n4275214\n4258547\n4241945\n4225407\n4208933\n4192524\n4176179\n4159898\n4143680\n4127525\n4111434\n4095405\n4079438\n4063534\n4047692\n4031911\n4016192\n4000535\n3984938\n3969402\n3953927\n3938512\n3923157\n3907862\n3892627\n3877451\n3862334\n3847277\n3832278\n3817337\n3802454\n3787630\n3772864\n3758155\n3743503\n3728908\n3714371\n3699890\n3685465\n3671097\n3656785\n3642528\n3628327\n3614182\n3600092\n3586056\n3572075\n3558149\n3544277\n3530459\n3516696\n3502985\n3489328\n3475725\n3462174\n3448676\n3435231\n3421839\n3408498\n3395210\n3381973\n3368788\n3355654\n3342572\n3329540\n3316560\n3303630\n3290750\n3277921\n3265141\n3252412\n3239732\n3227101\n3214520\n3201988\n3189504\n3177070\n3164683\n3152345\n3140056\n3127814\n3115620\n3103473\n3091374\n3079321\n3067316\n3055358\n3043446\n3031581\n3019762\n3007989\n2996262\n2984581\n2972945\n2961355\n2949809\n2938309\n2926854\n2915443\n2904077\n2892755\n2881477\n2870243\n2859053\n2847907\n2836804\n2825744\n2814728\n2803754\n2792823\n2781935\n2771089\n2760286\n2749525\n2738805\n2728128\n2717492\n2706897\n2696344\n2685832\n2675361\n2664931\n2654541\n2644192\n2633883\n2623615\n2613386\n2603198\n2593049\n2582939\n2572869\n2562839\n2552847\n2542895\n2532981\n2523106\n2513269\n2503471\n2493711\n2483988\n2474304\n2464658\n2455049\n2445478\n2435944\n2426447\n2416987\n2407564\n2398178\n2388828\n2379515\n2370238\n2360998\n2351793\n2342624\n2333491\n2324394\n2315332\n2306305\n2297314\n2288357\n2279436\n2270549\n2261697\n2252880\n2244097\n2235348\n2226633\n2217952\n2209305\n2200692\n2192112\n2183566\n2175053\n2166573\n2158127\n2149713\n2141332\n2132984\n2124668\n2116385\n2108134\n2099915\n2091728\n2083573\n2075450\n2067359\n2059299\n2051270\n2043273\n2035307\n2027372\n2019468\n2011595\n2003753\n1995941\n1988159\n1980408\n1972687\n1964997\n1957336\n1949705\n1942104\n1934532\n1926990\n1919478\n1911994\n1904540\n1897115\n1889719\n1882351\n1875013\n1867703\n1860421\n1853168\n1845943\n1838747\n1831578\n1824438\n1817325\n1810240\n1803182\n1796152\n1789150\n1782175\n1775227\n1768306\n1761412\n1754544\n1747704\n1740891\n1734103\n1727343\n1720609\n1713901\n1707219\n1700563\n1693933\n1687329\n1680751\n1674198\n1667671\n1661169\n1654693\n1648242\n1641816\n1635415\n1629039\n1622688\n1616362\n1610061\n1603784\n1597531\n1591303\n1585099\n1578919\n1572764\n1566632\n1560524\n1554440\n1548380\n1542344\n1536330\n1530341\n1524375\n1518432\n1512512\n1506615\n1500741\n1494891\n1489063\n1483257\n1477475\n1471714\n1465977\n1460261\n1454568\n1448898\n1443249\n1437622\n1432017\n1426435\n1420873\n1415334\n1409816\n1404320\n1398845\n1393391\n1387959\n1382548\n1377158\n1371789\n1366441\n1361113\n1355807\n1350521\n1345256\n1340011\n1334787\n1329583\n1324400\n1319236\n1314093\n1308970\n1303867\n1298783\n1293720\n1288676\n1283652\n1278648\n1273663\n1268697\n1263751\n1258824\n1253916\n1249028\n1244158\n1239308\n1234476\n1229663\n1224869\n1220094\n1215337\n1210599\n1205880\n1201178\n1196495\n1191831\n1187184\n1182556\n1177945\n1173353\n1168779\n1164222\n1159683\n1155162\n1150658\n1146172\n1141704\n1137253\n1132819\n1128403\n1124003\n1119621\n1115256\n1110908\n1106577\n1102263\n1097966\n1093685\n1089421\n1085174\n1080944\n1076729\n1072532\n1068350\n1064185\n1060036\n1055903\n1051787\n1047686\n1043602\n1039533\n1035480\n1031443\n1027422\n1023417\n1019427\n1015452\n1011494\n1007550\n1003622\n999709\n995812\n991930\n988062\n984210\n980373\n976551\n972744\n968951\n965174\n961411\n957663\n953929\n950210\n946506\n942816\n939140\n935479\n931832\n928199\n924580\n920975\n917385\n913808\n910246\n906697\n903162\n899641\n896134\n892640\n889160\n885693\n882240\n878801\n875375\n871962\n868563\n865176\n861803\n858443\n855097\n851763\n848442\n845135\n841840\n838558\n835288\n832032\n828788\n825557\n822338\n819132\n815939\n812758\n809589\n806433\n803289\n800157\n797038\n793930\n790835\n787752\n784681\n781622\n778574\n775539\n772515\n769504\n766504\n763515\n760539\n757574\n754620\n751678\n748748\n745829\n742921\n740025\n737139\n734266\n731403\n728551\n725711\n722882\n720064\n717256\n714460\n711675\n708900\n706136\n703383\n700641\n697910\n695189\n692478\n689779\n687089\n684411\n681743\n679085\n676437\n673800\n671173\n668556\n665950\n663354\n660768\n658191\n655625\n653069\n650523\n647987\n645461\n642944\n640438\n637941\n635454\n632977\n630509\n628051\n625602\n623163\n620734\n618314\n615903\n613502\n611110\n608728\n606354\n603990\n601636\n599290\n596954\n594626\n592308\n589999\n587699\n585408\n583125\n580852\n578587\n576332\n574085\n571847\n569617\n567396\n565184\n562981\n560786\n558600\n556422\n554253\n552092\n549940\n547796\n545660\n543533\n541413\n539303\n537200\n535106\n533020\n530942\n528872\n526810\n524756\n522710\n520672\n518642\n516620\n514606\n512600\n510602\n508611\n506628\n504653\n502685\n500726\n498773\n496829\n494892\n492963\n491041\n489126\n487219\n485320\n483428\n481543\n479666\n477796\n475933\n474077\n472229\n470388\n468554\n466728\n464908\n463096\n461290\n459492\n457700\n455916\n454138\n452368\n450604\n448848\n447098\n445355\n443618\n441889\n440166\n438450\n436741\n435038\n433342\n431652\n429970\n428293\n426624\n424960\n423304\n421653\n420009\n418372\n416741\n415116\n413498\n411886\n410280\n408680\n407087\n405500\n403919\n402344\n400776\n399213\n397657\n396107\n394562\n393024\n391492\n389966\n388445\n386931\n385422\n383920\n382423\n380932\n379447\n377968\n376494\n375026\n373564\n372108\n370657\n369212\n367773\n366339\n364910\n363488\n362071\n360659\n359253\n357852\n356457\n355068\n353683\n352304\n350931\n349563\n348200\n346843\n345490\n344143\n342802\n341465\n340134\n338808\n337487\n336171\n334861\n333555\n332255\n330959\n329669\n328384\n327104\n325828\n324558\n323293\n322032\n320777\n319526\n318281\n317040\n315804\n314572\n313346\n312124\n310908\n309695\n308488\n307285\n306087\n304894\n303705\n302521\n301342\n300167\n298997\n297831\n296670\n295514\n294361\n293214\n292071\n290932\n289798\n288668\n287543\n286422\n285305\n284193\n283085\n281981\n280882\n279787\n278696\n277609\n276527\n275449\n274375\n273305\n272240\n271178\n270121\n269068\n268019\n266974\n265933\n264897\n263864\n262835\n261810\n260790\n259773\n258760\n257751\n256747\n255746\n254749\n253755\n252766\n251781\n250799\n249821\n248847\n247877\n246911\n245948\n244989\n244034\n243083\n242135\n241191\n240251\n239314\n238381\n237452\n236526\n235604\n234685\n233770\n232859\n231951\n231047\n230146\n229249\n228355\n227465\n226578\n225695\n224815\n223938\n223065\n222196\n221329\n220467\n219607\n218751\n217898\n217048\n216202\n215359\n214520\n213683\n212850\n212021\n211194\n210371\n209550\n208733\n207920\n207109\n206302\n205497\n204696\n203898\n203103\n202311\n201523\n200737\n199954\n199175\n198398\n197625\n196854\n196087\n195323\n194561\n193802\n193047\n192294\n191545\n190798\n190054\n189313\n188575\n187840\n187107\n186378\n185651\n184928\n184207\n183489\n182773\n182061\n181351\n180644\n179940\n179238\n178539\n177843\n177150\n176459\n175771\n175086\n174403\n173723\n173046\n172372\n171699\n171030\n170363\n169699\n169038\n168379\n167722\n167068\n166417\n165768\n165122\n164478\n163837\n163198\n162562\n161928\n161297\n160668\n160042\n159418\n158796\n158177\n157560\n156946\n156334\n155725\n155118\n154513\n153910\n153310\n152713\n152117\n151524\n150934\n150345\n149759\n149175\n148594\n148014\n147437\n146862\n146290\n145719\n145151\n144585\n144022\n143460\n142901\n142344\n141789\n141236\n140686\n140137\n139591\n139046\n138504\n137964\n137427\n136891\n136357\n135825\n135296\n134768\n134243\n133720\n133198\n132679\n132162\n131647\n131133\n130622\n130113\n129606\n129100\n128597\n128096\n127596\n127099\n126603\n126110\n125618\n125128\n124640\n124155\n123671\n123188\n122708\n122230\n121753\n121279\n120806\n120335\n119866\n119398\n118933\n118469\n118007\n117547\n117089\n116632\n116178\n115725\n115274\n114824\n114377\n113931\n113486\n113044\n112603\n112164\n111727\n111291\n110858\n110425\n109995\n109566\n109139\n108713\n108290\n107867\n107447\n107028\n106611\n106195\n105781\n105369\n104958\n104549\n104141\n103735\n103331\n102928\n102526\n102127\n101729\n101332\n100937\n100543\n100151\n99761\n99372\n98985\n98599\n98214\n97831\n97450\n97070\n96692\n96315\n95939\n95565\n95193\n94821\n94452\n94084\n93717\n93351\n92987\n92625\n92264\n91904\n91546\n91189\n90833\n90479\n90127\n89775\n89425\n89077\n88729\n88383\n88039\n87696\n87354\n87013\n86674\n86336\n85999\n85664\n85330\n84997\n84666\n84336\n84007\n83680\n83353\n83028\n82705\n82382\n82061\n81741\n81423\n81105\n80789\n80474\n80160\n79848\n79536\n79226\n78917\n78610\n78303\n77998\n77694\n77391\n77089\n76789\n76489\n76191\n75894\n75598\n75304\n75010\n74718\n74426\n74136\n73847\n73559\n73272\n72987\n72702\n72419\n72136\n71855\n71575\n71296\n71018\n70741\n70465\n70191\n69917\n69644\n69373\n69102\n68833\n68565\n68297\n68031\n67766\n67502\n67238\n66976\n66715\n66455\n66196\n65938\n65681\n65425\n65170\n64916\n64663\n64411\n64159\n63909\n63660\n63412\n63165\n62918\n62673\n62429\n62185\n61943\n61702\n61461\n61221\n60983\n60745\n60508\n60272\n60037\n59803\n59570\n59338\n59106\n58876\n58646\n58418\n58190\n57963\n57737\n57512\n57288\n57065\n56842\n56620\n56400\n56180\n55961\n55743\n55525\n55309\n55093\n54878\n54665\n54451\n54239\n54028\n53817\n53607\n53398\n53190\n52983\n52776\n52570\n52365\n52161\n51958\n51755\n51554\n51353\n51152\n50953\n50754\n50556\n50359\n50163\n49967\n49773\n49579\n49385\n49193\n49001\n48810\n48620\n48430\n48241\n48053\n47866\n47679\n47493\n47308\n47124\n46940\n46757\n46575\n46393\n46212\n46032\n45853\n45674\n45496\n45318\n45142\n44966\n44790\n44616\n44442\n44269\n44096\n43924\n43753\n43582\n43412\n43243\n43075\n42907\n42739\n42573\n42407\n42241\n42077\n41913\n41749\n41587\n41424\n41263\n41102\n40942\n40782\n40623\n40465\n40307\n40150\n39993\n39837\n39682\n39527\n39373\n39220\n39067\n38915\n38763\n38612\n38461\n38311\n38162\n38013\n37865\n37717\n37570\n37424\n37278\n37133\n36988\n36844\n36700\n36557\n36414\n36272\n36131\n35990\n35850\n35710\n35571\n35432\n35294\n35156\n35019\n34883\n34747\n34611\n34476\n34342\n34208\n34075\n33942\n33810\n33678\n33547\n33416\n33285\n33156\n33026\n32898\n32769\n32642\n32514\n32388\n32261\n32136\n32010\n31886\n31761\n31637\n31514\n31391\n31269\n31147\n31025\n30905\n30784\n30664\n30544\n30425\n30307\n30189\n30071\n29954\n29837\n29721\n29605\n29489\n29374\n29260\n29146\n29032\n28919\n28806\n28694\n28582\n28471\n28360\n28249\n28139\n28029\n27920\n27811\n27703\n27595\n27487\n27380\n27273\n27167\n27061\n26955\n26850\n26746\n26641\n26537\n26434\n26331\n26228\n26126\n26024\n25923\n25822\n25721\n25621\n25521\n25421\n25322\n25224\n25125\n25027\n24930\n24832\n24736\n24639\n24543\n24447\n24352\n24257\n24163\n24068\n23975\n23881\n23788\n23695\n23603\n23511\n23419\n23328\n23237\n23146\n23056\n22966\n22877\n22788\n22699\n22610\n22522\n22434\n22347\n22260\n22173\n22086\n22000\n21915\n21829\n21744\n21659\n21575\n21491\n21407\n21323\n21240\n21158\n21075\n20993\n20911\n20830\n20748\n20667\n20587\n20507\n20427\n20347\n20268\n20189\n20110\n20032\n19953\n19876\n19798\n19721\n19644\n19568\n19491\n19415\n19340\n19264\n19189\n19114\n19040\n18965\n18892\n18818\n18745\n18671\n18599\n18526\n18454\n18382\n18310\n18239\n18168\n18097\n18026\n17956\n17886\n17816\n17747\n17678\n17609\n17540\n17472\n17404\n17336\n17268\n17201\n17134\n17067\n17001\n16934\n16868\n16802\n16737\n16672\n16607\n16542\n16477\n16413\n16349\n16286\n16222\n16159\n16096\n16033\n15971\n15908\n15846\n15784\n15723\n15662\n15601\n15540\n15479\n15419\n15359\n15299\n15239\n15180\n15121\n15062\n15003\n14944\n14886\n14828\n14770\n14713\n14655\n14598\n14541\n14485\n14428\n14372\n14316\n14260\n14204\n14149\n14094\n14039\n13984\n13930\n13875\n13821\n13767\n13714\n13660\n13607\n13554\n13501\n13449\n13396\n13344\n13292\n13240\n13188\n13137\n13086\n13035\n12984\n12933\n12883\n12833\n12783\n12733\n12683\n12634\n12584\n12535\n12487\n12438\n12389\n12341\n12293\n12245\n12197\n12150\n12102\n12055\n12008\n11961\n11915\n11868\n11822\n11776\n11730\n11684\n11639\n11593\n11548\n11503\n11458\n11414\n11369\n11325\n11281\n11237\n11193\n11149\n11106\n11062\n11019\n10976\n10934\n10891\n10849\n10806\n10764\n10722\n10680\n10639\n10597\n10556\n10515\n10474\n10433\n10392\n10352\n10311\n10271\n10231\n10191\n10151\n10112\n10072\n10033\n9994\n9955\n9916\n9878\n9839\n9801\n9763\n9725\n9687\n9649\n9611\n9574\n9536\n9499\n9462\n9425\n9389\n9352\n9316\n9279\n9243\n9207\n9171\n9135\n9100\n9064\n9029\n8994\n8959\n8924\n8889\n8854\n8820\n8785\n8751\n8717\n8683\n8649\n8615\n8582\n8548\n8515\n8482\n8449\n8416\n8383\n8350\n8318\n8285\n8253\n8221\n8189\n8157\n8125\n8093\n8062\n8030\n7999\n7968\n7937\n7906\n7875\n7844\n7814\n7783\n7753\n7723\n7693\n7663\n7633\n7603\n7573\n7544\n7515\n7485\n7456\n7427\n7398\n7369\n7340\n7312\n7283\n7255\n7227\n7198\n7170\n7142\n7115\n7087\n7059\n7032\n7004\n6977\n6950\n6923\n6896\n6869\n6842\n6815\n6789\n6762\n6736\n6710\n6684\n6658\n6632\n6606\n6580\n6554\n6529\n6503\n6478\n6453\n6428\n6402\n6378\n6353\n6328\n6303\n6279\n6254\n6230\n6205\n6181\n6157\n6133\n6109\n6085\n6062\n6038\n6015\n5991\n5968\n5944\n5921\n5898\n5875\n5852\n5830\n5807\n5784\n5762\n5739\n5717\n5694\n5672\n5650\n5628\n5606\n5584\n5563\n5541\n5519\n5498\n5476\n5455\n5434\n5413\n5391\n5370\n5349\n5329\n5308\n5287\n5267\n5246\n5226\n5205\n5185\n5165\n5145\n5124\n5104\n5085\n5065\n5045\n5025\n5006\n4986\n4967\n4947\n4928\n4909\n4890\n4871\n4852\n4833\n4814\n4795\n4777\n4758\n4739\n4721\n4702\n4684\n4666\n4648\n4630\n4612\n4594\n4576\n4558\n4540\n4522\n4505\n4487\n4470\n4452\n4435\n4418\n4400\n4383\n4366\n4349\n4332\n4315\n4298\n4282\n4265\n4248\n4232\n4215\n4199\n4182\n4166\n4150\n4134\n4118\n4102\n4086\n4070\n4054\n4038\n4022\n4007\n3991\n3975\n3960\n3944\n3929\n3914\n3898\n3883\n3868\n3853\n3838\n3823\n3808\n3793\n3779\n3764\n3749\n3735\n3720\n3705\n3691\n3677\n3662\n3648\n3634\n3620\n3606\n3591\n3577\n3564\n3550\n3536\n3522\n3508\n3495\n3481\n3467\n3454\n3440\n3427\n3414\n3400\n3387\n3374\n3361\n3348\n3335\n3322\n3309\n3296\n3283\n3270\n3257\n3245\n3232\n3219\n3207\n3194\n3182\n3169\n3157\n3145\n3133\n3120\n3108\n3096\n3084\n3072\n3060\n3048\n3036\n3024\n3013\n3001\n2989\n2977\n2966\n2954\n2943\n2931\n2920\n2908\n2897\n2886\n2875\n2863\n2852\n2841\n2830\n2819\n2808\n2797\n2786\n2775\n2764\n2754\n2743\n2732\n2722\n2711\n2700\n2690\n2679\n2669\n2659\n2648\n2638\n2628\n2617\n2607\n2597\n2587\n2577\n2567\n2557\n2547\n2537\n2527\n2517\n2507\n2497\n2488\n2478\n2468\n2459\n2449\n2440\n2430\n2421\n2411\n2402\n2392\n2383\n2374\n2365\n2355\n2346\n2337\n2328\n2319\n2310\n2301\n2292\n2283\n2274\n2265\n2256\n2247\n2239\n2230\n2221\n2213\n2204\n2195\n2187\n2178\n2170\n2161\n2153\n2145\n2136\n2128\n2120\n2111\n2103\n2095\n2087\n2079\n2070\n2062\n2054\n2046\n2038\n2030\n2023\n2015\n2007\n1999\n1991\n1983\n1976\n1968\n1960\n1953\n1945\n1937\n1930\n1922\n1915\n1907\n1900\n1893\n1885\n1878\n1871\n1863\n1856\n1849\n1842\n1834\n1827\n1820\n1813\n1806\n1799\n1792\n1785\n1778\n1771\n1764\n1757\n1750\n1744\n1737\n1730\n1723\n1716\n1710\n1703\n1696\n1690\n1683\n1677\n1670\n1664\n1657\n1651\n1644\n1638\n1631\n1625\n1619\n1612\n1606\n1600\n1594\n1587\n1581\n1575\n1569\n1563\n1557\n1551\n1545\n1539\n1533\n1527\n1521\n1515\n1509\n1503\n1497\n1491\n1485\n1480\n1474\n1468\n1462\n1457\n1451\n1445\n1440\n1434\n1429\n1423\n1417\n1412\n1406\n1401\n1395\n1390\n1385\n1379\n1374\n1368\n1363\n1358\n1353\n1347\n1342\n1337\n1332\n1326\n1321\n1316\n1311\n1306\n1301\n1296\n1291\n1286\n1281\n1276\n1271\n1266\n1261\n1256\n1251\n1246\n1241\n1236\n1232\n1227\n1222\n1217\n1212\n1208\n1203\n1198\n1194\n1189\n1184\n1180\n1175\n1171\n1166\n1161\n1157\n1152\n1148\n1143\n1139\n1135\n1130\n1126\n1121\n1117\n1113\n1108\n1104\n1100\n1095\n1091\n1087\n1083\n1078\n1074\n1070\n1066\n1062\n1057\n1053\n1049\n1045\n1041\n1037\n1033\n1029\n1025\n1021\n1017\n1013\n1009\n1005\n1001\n997\n993\n990\n986\n982\n978\n974\n970\n967\n963\n959\n955\n952\n948\n944\n941\n937\n933\n930\n926\n922\n919\n915\n912\n908\n905\n901\n897\n894\n890\n887\n884\n880\n877\n873\n870\n866\n863\n860\n856\n853\n850\n846\n843\n840\n837\n833\n830\n827\n824\n820\n817\n814\n811\n808\n804\n801\n798\n795\n792\n789\n786\n783\n780\n777\n774\n771\n768\n765\n762\n759\n756\n753\n750\n747\n744\n741\n738\n735\n733\n730\n727\n724\n721\n718\n716\n713\n710\n707\n704\n702\n699\n696\n694\n691\n688\n685\n683\n680\n677\n675\n672\n670\n667\n664\n662\n659\n657\n654\n652\n649\n646\n644\n641\n639\n636\n634\n631\n629\n627\n624\n622\n619\n617\n614\n612\n610\n607\n605\n603\n600\n598\n596\n593\n591\n589\n586\n584\n582\n579\n577\n575\n573\n570\n568\n566\n564\n562\n559\n557\n555\n553\n551\n549\n546\n544\n542\n540\n538\n536\n534\n532\n530\n528\n526\n523\n521\n519\n517\n515\n513\n511\n509\n507\n505\n503\n501\n500\n498\n496\n494\n492\n490\n488\n486\n484\n482\n480\n479\n477\n475\n473\n471\n469\n467\n466\n464\n462\n460\n458\n457\n455\n453\n451\n450\n448\n446\n444\n443\n441\n439\n437\n436\n434\n432\n431\n429\n427\n426\n424\n422\n421\n419\n417\n416\n414\n413\n411\n409\n408\n406\n405\n403\n401\n400\n398\n397\n395\n394\n392\n391\n389\n388\n386\n384\n383\n382\n380\n379\n377\n376\n374\n373\n371\n370\n368\n367\n365\n364\n363\n361\n360\n358\n357\n356\n354\n353\n351\n350\n349\n347\n346\n345\n343\n342\n341\n339\n338\n337\n335\n334\n333\n331\n330\n329\n328\n326\n325\n324\n323\n321\n320\n319\n318\n316\n315\n314\n313\n311\n310\n309\n308\n307\n305\n304\n303\n302\n301\n299\n298\n297\n296\n295\n294\n293\n291\n290\n289\n288\n287\n286\n285\n284\n282\n281\n280\n279\n278\n277\n276\n275\n274\n273\n272\n271\n269\n268\n267\n266\n265\n264\n263\n262\n261\n260\n259\n258\n257\n256\n255\n254\n253\n252\n251\n250\n249\n248\n247\n246\n245\n244\n243\n242\n242\n241\n240\n239\n238\n237\n236\n235\n234\n233\n232\n231\n230\n230\n229\n228\n227\n226\n225\n224\n223\n223\n222\n221\n220\n219\n218\n217\n217\n216\n215\n214\n213\n212\n212\n211\n210\n209\n208\n207\n207\n206\n205\n204\n203\n203\n202\n201\n200\n199\n199\n198\n197\n196\n196\n195\n194\n193\n193\n192\n191\n190\n190\n189\n188\n187\n187\n186\n185\n184\n184\n183\n182\n182\n181\n180\n180\n179\n178\n177\n177\n176\n175\n175\n174\n173\n173\n172\n171\n171\n170\n169\n169\n168\n167\n167\n166\n165\n165\n164\n163\n163\n162\n162\n161\n160\n160\n159\n158\n158\n157\n157\n156\n155\n155\n154\n154\n153\n152\n152\n151\n151\n150\n149\n149\n148\n148\n147\n147\n146\n145\n145\n144\n144\n143\n143\n142\n141\n141\n140\n140\n139\n139\n138\n138\n137\n137\n136\n135\n135\n134\n134\n133\n133\n132\n132\n131\n131\n130\n130\n129\n129\n128\n128\n127\n127\n126\n126\n125\n125\n124\n124\n123\n123\n122\n122\n121\n121\n121\n120\n120\n119\n119\n118\n118\n117\n117\n116\n116\n115\n115\n115\n114\n114\n113\n113\n112\n112\n111\n111\n111\n110\n110\n109\n109\n108\n108\n108\n107\n107\n106\n106\n106\n105\n105\n104\n104\n103\n103\n103\n102\n102\n101\n101\n101\n100\n100\n100\n99\n99\n98\n98\n98\n97\n97\n96\n96\n96\n95\n95\n95\n94\n94\n93\n93\n93\n92\n92\n92\n91\n91\n91\n90\n90\n90\n89\n89\n89\n88\n88\n87\n87\n87\n86\n86\n86\n85\n85\n85\n84\n84\n84\n83\n83\n83\n83\n82\n82\n82\n81\n81\n81\n80\n80\n80\n79\n79\n79\n78\n78\n78\n78\n77\n77\n77\n76\n76\n76\n75\n75\n75\n75\n74\n74\n74\n73\n73\n73\n73\n72\n72\n72\n71\n71\n71\n71\n70\n70\n70\n69\n69\n69\n69\n68\n68\n68\n68\n67\n67\n67\n67\n66\n66\n66\n66\n65\n65\n65\n65\n64\n64\n64\n64\n63\n63\n63\n63\n62\n62\n62\n62\n61\n61\n61\n61\n60\n60\n60\n60\n59\n59\n59\n59\n59\n58\n58\n58\n58\n57\n57\n57\n57\n56\n56\n56\n56\n56\n55\n55\n55\n55\n55\n54\n54\n54\n54\n53\n53\n53\n53\n53\n52\n52\n52\n52\n52\n51\n51\n51\n51\n51\n50\n50\n50\n50\n50\n49\n49\n49\n49\n49\n49\n48\n48\n48\n48\n48\n47\n47\n47\n47\n47\n46\n46\n46\n46\n46\n46\n45\n45\n45\n45\n45\n45\n44\n44\n44\n44\n44\n43\n43\n43\n43\n43\n43\n42\n42\n42\n42\n42\n42\n41\n41\n41\n41\n41\n41\n41\n40\n40\n40\n40\n40\n40\n39\n39\n39\n39\n39\n39\n39\n38\n38\n38\n38\n38\n38\n37\n37\n37\n37\n37\n37\n37\n36\n36\n36\n36\n36\n36\n36\n35\n35\n35\n35\n35\n35\n35\n35\n34\n34\n34\n34\n34\n34\n34\n33\n33\n33\n33\n33\n33\n33\n33\n32\n32\n32\n32\n32\n32\n32\n32\n31\n31\n31\n31\n31\n31\n31\n31\n30\n30\n30\n30\n30\n30\n30\n30\n30\n29\n29\n29\n29\n29\n29\n29\n29\n29\n28\n28\n28\n28\n28\n28\n28\n28\n28\n27\n27\n27\n27\n27\n27\n27\n27\n27\n26\n26\n26\n26\n26\n26\n26\n26\n26\n26\n25\n25\n25\n25\n25\n25\n25\n25\n25\n25\n24\n24\n24\n24\n24\n24\n24\n24\n24\n24\n24\n23\n23\n23\n23\n23\n23\n23\n23\n23\n23\n23\n22\n22\n22\n22\n22\n22\n22\n22\n22\n22\n22\n22\n21\n21\n21\n21\n21\n21\n21\n21\n21\n21\n21\n21\n20\n20\n20\n20\n20\n20\n20\n20\n20\n20\n20\n20\n20\n19\n19\n19\n19\n19\n19\n19\n19\n19\n19\n19\n19\n19\n18\n18\n18\n18\n18\n18\n18\n18\n18\n18\n18\n18\n18\n18\n17\n17\n17\n17\n17\n17\n17\n17\n17\n17\n17\n17\n17\n17\n17\n17\n16\n16\n16\n16\n16\n16\n16\n16\n16\n16\n16\n16\n16\n16\n16\n16\n15\n15\n15\n15\n15\n15\n15\n15\n15\n15\n15\n15\n15\n15\n15\n15\n15\n14\n14\n14\n14\n14\n14\n14\n14\n14\n14\n14\n14\n14\n14\n14\n14\n14\n14\n13\n13\n13\n13\n13\n13\n13\n13\n13\n13\n13\n13\n13\n13\n13\n13\n13\n13\n13\n13\n12\n12\n12\n12\n12\n12\n12\n12\n12\n12\n12\n12\n12\n12\n12\n12\n12\n12\n12\n12\n12\n11\n11\n11\n11\n11\n11\n11\n11\n11\n11\n11\n11\n11\n11\n11\n11\n11\n11\n11\n11\n11\n11\n11\n10\n10\n10\n10\n10\n10\n10\n10\n10\n10\n10\n10\n10\n10\n10\n10\n10\n10\n10\n10\n10\n10\n10\n10\n10\n10\n9\n9\n9\n9\n9\n9\n9\n9\n9\n9\n9\n9\n9\n9\n9\n9\n9\n9\n9\n9\n9\n9\n9\n9\n9\n9\n9\n9\n8\n8\n8\n8\n8\n8\n8\n8\n8\n8\n8\n8\n8\n8\n8\n8\n8\n8\n8\n8\n8\n8\n8\n8\n8\n8\n8\n8\n8\n8\n8\n8\n7\n7\n7\n7\n7\n7\n7\n7\n7\n7\n7\n7\n7\n7\n7\n7\n7\n7\n7\n7\n7\n7\n7\n7\n7\n7\n7\n7\n7\n7\n7\n7\n7\n7\n7\n7\n7\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n')
FILES['tables/gelu.txt'] = ('896d56e01ca498795d6e3becb8539c78697fe3d2d4c44f3915a921110bc21679', '# gelu table\n# G[k] = round(2^16 * x * Phi(x)), x = -8 + k/256, k = 0 .. 4096\n# 4097 values, one per line, generated by make_tables.py\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-5\n-5\n-5\n-5\n-5\n-5\n-5\n-5\n-5\n-5\n-5\n-5\n-6\n-6\n-6\n-6\n-6\n-6\n-6\n-6\n-6\n-6\n-6\n-7\n-7\n-7\n-7\n-7\n-7\n-7\n-7\n-7\n-8\n-8\n-8\n-8\n-8\n-8\n-8\n-8\n-9\n-9\n-9\n-9\n-9\n-9\n-9\n-10\n-10\n-10\n-10\n-10\n-10\n-10\n-11\n-11\n-11\n-11\n-11\n-11\n-12\n-12\n-12\n-12\n-12\n-13\n-13\n-13\n-13\n-13\n-14\n-14\n-14\n-14\n-14\n-15\n-15\n-15\n-15\n-15\n-16\n-16\n-16\n-16\n-17\n-17\n-17\n-17\n-18\n-18\n-18\n-19\n-19\n-19\n-19\n-20\n-20\n-20\n-20\n-21\n-21\n-21\n-22\n-22\n-22\n-23\n-23\n-23\n-24\n-24\n-24\n-25\n-25\n-25\n-26\n-26\n-27\n-27\n-27\n-28\n-28\n-29\n-29\n-29\n-30\n-30\n-31\n-31\n-32\n-32\n-32\n-33\n-33\n-34\n-34\n-35\n-35\n-36\n-36\n-37\n-37\n-38\n-38\n-39\n-39\n-40\n-41\n-41\n-42\n-42\n-43\n-43\n-44\n-45\n-45\n-46\n-47\n-47\n-48\n-49\n-49\n-50\n-51\n-51\n-52\n-53\n-53\n-54\n-55\n-56\n-56\n-57\n-58\n-59\n-59\n-60\n-61\n-62\n-63\n-64\n-64\n-65\n-66\n-67\n-68\n-69\n-70\n-71\n-72\n-73\n-74\n-74\n-75\n-76\n-77\n-78\n-80\n-81\n-82\n-83\n-84\n-85\n-86\n-87\n-88\n-89\n-91\n-92\n-93\n-94\n-95\n-97\n-98\n-99\n-100\n-102\n-103\n-104\n-106\n-107\n-108\n-110\n-111\n-113\n-114\n-115\n-117\n-118\n-120\n-121\n-123\n-124\n-126\n-128\n-129\n-131\n-132\n-134\n-136\n-137\n-139\n-141\n-143\n-144\n-146\n-148\n-150\n-152\n-154\n-156\n-157\n-159\n-161\n-163\n-165\n-167\n-169\n-171\n-173\n-176\n-178\n-180\n-182\n-184\n-187\n-189\n-191\n-193\n-196\n-198\n-200\n-203\n-205\n-208\n-210\n-213\n-215\n-218\n-220\n-223\n-226\n-228\n-231\n-234\n-236\n-239\n-242\n-245\n-248\n-250\n-253\n-256\n-259\n-262\n-265\n-268\n-272\n-275\n-278\n-281\n-284\n-288\n-291\n-294\n-298\n-301\n-304\n-308\n-311\n-315\n-318\n-322\n-326\n-329\n-333\n-337\n-341\n-344\n-348\n-352\n-356\n-360\n-364\n-368\n-372\n-376\n-381\n-385\n-389\n-393\n-398\n-402\n-407\n-411\n-415\n-420\n-425\n-429\n-434\n-439\n-443\n-448\n-453\n-458\n-463\n-468\n-473\n-478\n-483\n-488\n-494\n-499\n-504\n-509\n-515\n-520\n-526\n-531\n-537\n-543\n-548\n-554\n-560\n-566\n-572\n-578\n-584\n-590\n-596\n-602\n-608\n-615\n-621\n-627\n-634\n-640\n-647\n-654\n-660\n-667\n-674\n-681\n-688\n-695\n-702\n-709\n-716\n-723\n-731\n-738\n-745\n-753\n-760\n-768\n-776\n-783\n-791\n-799\n-807\n-815\n-823\n-831\n-839\n-847\n-856\n-864\n-873\n-881\n-890\n-898\n-907\n-916\n-925\n-934\n-943\n-952\n-961\n-970\n-979\n-989\n-998\n-1008\n-1017\n-1027\n-1037\n-1047\n-1057\n-1066\n-1077\n-1087\n-1097\n-1107\n-1117\n-1128\n-1138\n-1149\n-1160\n-1170\n-1181\n-1192\n-1203\n-1214\n-1225\n-1237\n-1248\n-1259\n-1271\n-1282\n-1294\n-1306\n-1318\n-1329\n-1341\n-1354\n-1366\n-1378\n-1390\n-1403\n-1415\n-1428\n-1441\n-1453\n-1466\n-1479\n-1492\n-1505\n-1519\n-1532\n-1545\n-1559\n-1572\n-1586\n-1600\n-1614\n-1628\n-1642\n-1656\n-1670\n-1684\n-1699\n-1713\n-1728\n-1743\n-1758\n-1772\n-1787\n-1803\n-1818\n-1833\n-1848\n-1864\n-1880\n-1895\n-1911\n-1927\n-1943\n-1959\n-1975\n-1991\n-2008\n-2024\n-2041\n-2058\n-2074\n-2091\n-2108\n-2125\n-2143\n-2160\n-2177\n-2195\n-2212\n-2230\n-2248\n-2266\n-2284\n-2302\n-2320\n-2339\n-2357\n-2376\n-2394\n-2413\n-2432\n-2451\n-2470\n-2489\n-2509\n-2528\n-2547\n-2567\n-2587\n-2607\n-2627\n-2647\n-2667\n-2687\n-2707\n-2728\n-2748\n-2769\n-2790\n-2811\n-2832\n-2853\n-2874\n-2895\n-2917\n-2938\n-2960\n-2982\n-3004\n-3026\n-3048\n-3070\n-3092\n-3115\n-3137\n-3160\n-3183\n-3205\n-3228\n-3252\n-3275\n-3298\n-3321\n-3345\n-3368\n-3392\n-3416\n-3440\n-3464\n-3488\n-3512\n-3537\n-3561\n-3586\n-3610\n-3635\n-3660\n-3685\n-3710\n-3735\n-3760\n-3786\n-3811\n-3837\n-3863\n-3888\n-3914\n-3940\n-3966\n-3993\n-4019\n-4045\n-4072\n-4098\n-4125\n-4152\n-4179\n-4206\n-4233\n-4260\n-4288\n-4315\n-4343\n-4370\n-4398\n-4426\n-4453\n-4481\n-4510\n-4538\n-4566\n-4594\n-4623\n-4651\n-4680\n-4709\n-4737\n-4766\n-4795\n-4824\n-4854\n-4883\n-4912\n-4942\n-4971\n-5001\n-5030\n-5060\n-5090\n-5120\n-5150\n-5180\n-5210\n-5240\n-5270\n-5301\n-5331\n-5362\n-5392\n-5423\n-5454\n-5485\n-5515\n-5546\n-5577\n-5609\n-5640\n-5671\n-5702\n-5734\n-5765\n-5796\n-5828\n-5860\n-5891\n-5923\n-5955\n-5987\n-6018\n-6050\n-6082\n-6114\n-6146\n-6179\n-6211\n-6243\n-6275\n-6308\n-6340\n-6372\n-6405\n-6437\n-6470\n-6502\n-6535\n-6567\n-6600\n-6633\n-6665\n-6698\n-6731\n-6764\n-6797\n-6829\n-6862\n-6895\n-6928\n-6961\n-6994\n-7027\n-7060\n-7093\n-7126\n-7159\n-7192\n-7225\n-7258\n-7291\n-7324\n-7357\n-7390\n-7423\n-7456\n-7489\n-7522\n-7555\n-7587\n-7620\n-7653\n-7686\n-7719\n-7752\n-7785\n-7818\n-7850\n-7883\n-7916\n-7948\n-7981\n-8014\n-8046\n-8079\n-8111\n-8144\n-8176\n-8208\n-8241\n-8273\n-8305\n-8337\n-8369\n-8401\n-8433\n-8465\n-8497\n-8529\n-8560\n-8592\n-8623\n-8655\n-8686\n-8717\n-8749\n-8780\n-8811\n-8842\n-8873\n-8903\n-8934\n-8964\n-8995\n-9025\n-9055\n-9086\n-9116\n-9146\n-9175\n-9205\n-9234\n-9264\n-9293\n-9322\n-9351\n-9380\n-9409\n-9438\n-9466\n-9494\n-9523\n-9551\n-9579\n-9606\n-9634\n-9661\n-9689\n-9716\n-9743\n-9769\n-9796\n-9822\n-9849\n-9875\n-9901\n-9926\n-9952\n-9977\n-10002\n-10027\n-10052\n-10077\n-10101\n-10125\n-10149\n-10173\n-10196\n-10219\n-10243\n-10265\n-10288\n-10310\n-10333\n-10354\n-10376\n-10398\n-10419\n-10440\n-10461\n-10481\n-10501\n-10521\n-10541\n-10560\n-10579\n-10598\n-10617\n-10635\n-10653\n-10671\n-10689\n-10706\n-10723\n-10740\n-10756\n-10772\n-10788\n-10803\n-10818\n-10833\n-10848\n-10862\n-10876\n-10889\n-10903\n-10916\n-10928\n-10940\n-10952\n-10964\n-10975\n-10986\n-10997\n-11007\n-11017\n-11026\n-11035\n-11044\n-11053\n-11061\n-11068\n-11076\n-11082\n-11089\n-11095\n-11101\n-11106\n-11111\n-11116\n-11120\n-11124\n-11127\n-11130\n-11133\n-11135\n-11137\n-11138\n-11139\n-11139\n-11139\n-11139\n-11138\n-11137\n-11135\n-11133\n-11130\n-11127\n-11123\n-11120\n-11115\n-11110\n-11105\n-11099\n-11093\n-11086\n-11079\n-11071\n-11063\n-11054\n-11045\n-11035\n-11025\n-11014\n-11003\n-10991\n-10979\n-10966\n-10953\n-10939\n-10925\n-10910\n-10895\n-10879\n-10863\n-10846\n-10828\n-10810\n-10792\n-10773\n-10753\n-10733\n-10712\n-10691\n-10669\n-10646\n-10623\n-10600\n-10576\n-10551\n-10526\n-10500\n-10474\n-10447\n-10419\n-10391\n-10362\n-10333\n-10303\n-10272\n-10241\n-10209\n-10177\n-10144\n-10110\n-10076\n-10041\n-10006\n-9970\n-9933\n-9895\n-9857\n-9819\n-9780\n-9740\n-9699\n-9658\n-9616\n-9574\n-9531\n-9487\n-9442\n-9397\n-9352\n-9305\n-9258\n-9210\n-9162\n-9113\n-9063\n-9013\n-8962\n-8910\n-8857\n-8804\n-8750\n-8696\n-8641\n-8585\n-8528\n-8471\n-8413\n-8354\n-8294\n-8234\n-8174\n-8112\n-8050\n-7987\n-7923\n-7859\n-7794\n-7728\n-7661\n-7594\n-7526\n-7457\n-7388\n-7317\n-7247\n-7175\n-7102\n-7029\n-6955\n-6881\n-6805\n-6729\n-6652\n-6575\n-6496\n-6417\n-6337\n-6257\n-6175\n-6093\n-6010\n-5927\n-5842\n-5757\n-5671\n-5585\n-5497\n-5409\n-5320\n-5230\n-5140\n-5048\n-4956\n-4863\n-4770\n-4675\n-4580\n-4484\n-4388\n-4290\n-4192\n-4093\n-3993\n-3892\n-3791\n-3689\n-3586\n-3482\n-3377\n-3272\n-3166\n-3059\n-2951\n-2843\n-2733\n-2623\n-2512\n-2401\n-2288\n-2175\n-2061\n-1946\n-1830\n-1714\n-1597\n-1479\n-1360\n-1240\n-1120\n-998\n-876\n-754\n-630\n-506\n-380\n-254\n-128\n0\n128\n258\n388\n518\n650\n782\n916\n1050\n1184\n1320\n1456\n1593\n1731\n1870\n2010\n2150\n2291\n2433\n2576\n2719\n2864\n3009\n3155\n3301\n3449\n3597\n3746\n3896\n4047\n4198\n4350\n4503\n4657\n4812\n4967\n5123\n5280\n5438\n5596\n5756\n5916\n6077\n6238\n6401\n6564\n6728\n6892\n7058\n7224\n7391\n7559\n7727\n7897\n8067\n8238\n8409\n8582\n8755\n8929\n9103\n9279\n9455\n9632\n9809\n9988\n10167\n10347\n10527\n10709\n10891\n11074\n11257\n11441\n11627\n11812\n11999\n12186\n12374\n12563\n12752\n12942\n13133\n13325\n13517\n13710\n13904\n14098\n14294\n14490\n14686\n14883\n15081\n15280\n15479\n15679\n15880\n16082\n16284\n16487\n16690\n16894\n17099\n17305\n17511\n17718\n17926\n18134\n18343\n18552\n18763\n18974\n19185\n19397\n19610\n19824\n20038\n20253\n20468\n20684\n20901\n21119\n21337\n21555\n21774\n21994\n22215\n22436\n22658\n22880\n23103\n23327\n23551\n23776\n24001\n24227\n24454\n24681\n24909\n25137\n25366\n25596\n25826\n26057\n26288\n26520\n26753\n26986\n27219\n27453\n27688\n27923\n28159\n28395\n28632\n28870\n29108\n29346\n29585\n29825\n30065\n30306\n30547\n30789\n31031\n31274\n31517\n31761\n32005\n32250\n32495\n32741\n32987\n33234\n33481\n33729\n33977\n34226\n34475\n34725\n34975\n35226\n35477\n35728\n35981\n36233\n36486\n36739\n36993\n37247\n37502\n37757\n38013\n38269\n38525\n38782\n39039\n39297\n39555\n39814\n40073\n40332\n40592\n40852\n41113\n41374\n41635\n41897\n42159\n42422\n42684\n42948\n43211\n43475\n43740\n44005\n44270\n44535\n44801\n45067\n45334\n45601\n45868\n46136\n46404\n46672\n46940\n47209\n47479\n47748\n48018\n48288\n48559\n48830\n49101\n49372\n49644\n49916\n50188\n50461\n50734\n51007\n51281\n51555\n51829\n52103\n52378\n52653\n52928\n53203\n53479\n53755\n54031\n54307\n54584\n54861\n55138\n55416\n55694\n55971\n56250\n56528\n56807\n57085\n57365\n57644\n57923\n58203\n58483\n58763\n59043\n59324\n59605\n59886\n60167\n60448\n60730\n61011\n61293\n61575\n61858\n62140\n62423\n62705\n62988\n63271\n63555\n63838\n64122\n64405\n64689\n64973\n65258\n65542\n65826\n66111\n66396\n66681\n66966\n67251\n67536\n67822\n68107\n68393\n68678\n68964\n69250\n69537\n69823\n70109\n70396\n70682\n70969\n71255\n71542\n71829\n72116\n72403\n72691\n72978\n73265\n73553\n73840\n74128\n74415\n74703\n74991\n75279\n75567\n75855\n76143\n76431\n76719\n77007\n77296\n77584\n77872\n78161\n78449\n78738\n79026\n79315\n79604\n79892\n80181\n80470\n80758\n81047\n81336\n81625\n81914\n82203\n82492\n82781\n83069\n83358\n83647\n83936\n84225\n84514\n84803\n85092\n85381\n85670\n85959\n86248\n86537\n86826\n87115\n87404\n87693\n87982\n88271\n88560\n88849\n89138\n89427\n89715\n90004\n90293\n90582\n90871\n91159\n91448\n91737\n92025\n92314\n92602\n92891\n93179\n93468\n93756\n94044\n94333\n94621\n94909\n95197\n95486\n95774\n96062\n96350\n96638\n96925\n97213\n97501\n97789\n98076\n98364\n98652\n98939\n99226\n99514\n99801\n100088\n100375\n100663\n100950\n101237\n101523\n101810\n102097\n102384\n102670\n102957\n103243\n103530\n103816\n104102\n104388\n104674\n104960\n105246\n105532\n105818\n106103\n106389\n106674\n106960\n107245\n107530\n107816\n108101\n108386\n108671\n108955\n109240\n109525\n109809\n110094\n110378\n110662\n110946\n111231\n111515\n111798\n112082\n112366\n112649\n112933\n113216\n113500\n113783\n114066\n114349\n114632\n114915\n115198\n115480\n115763\n116045\n116327\n116610\n116892\n117174\n117456\n117737\n118019\n118301\n118582\n118864\n119145\n119426\n119707\n119988\n120269\n120550\n120830\n121111\n121391\n121672\n121952\n122232\n122512\n122792\n123072\n123352\n123631\n123911\n124190\n124469\n124748\n125028\n125307\n125585\n125864\n126143\n126421\n126700\n126978\n127256\n127534\n127812\n128090\n128368\n128646\n128923\n129201\n129478\n129755\n130032\n130309\n130586\n130863\n131140\n131416\n131693\n131969\n132245\n132521\n132797\n133073\n133349\n133625\n133901\n134176\n134451\n134727\n135002\n135277\n135552\n135827\n136102\n136376\n136651\n136925\n137200\n137474\n137748\n138022\n138296\n138570\n138844\n139117\n139391\n139664\n139937\n140211\n140484\n140757\n141030\n141302\n141575\n141848\n142120\n142393\n142665\n142937\n143209\n143481\n143753\n144025\n144296\n144568\n144840\n145111\n145382\n145653\n145925\n146196\n146466\n146737\n147008\n147279\n147549\n147820\n148090\n148360\n148630\n148900\n149170\n149440\n149710\n149980\n150249\n150519\n150788\n151057\n151327\n151596\n151865\n152134\n152403\n152671\n152940\n153209\n153477\n153746\n154014\n154282\n154550\n154819\n155087\n155354\n155622\n155890\n156158\n156425\n156693\n156960\n157227\n157495\n157762\n158029\n158296\n158563\n158830\n159096\n159363\n159630\n159896\n160163\n160429\n160695\n160961\n161227\n161494\n161759\n162025\n162291\n162557\n162823\n163088\n163354\n163619\n163885\n164150\n164415\n164680\n164945\n165210\n165475\n165740\n166005\n166270\n166534\n166799\n167063\n167328\n167592\n167857\n168121\n168385\n168649\n168913\n169177\n169441\n169705\n169969\n170232\n170496\n170760\n171023\n171287\n171550\n171813\n172077\n172340\n172603\n172866\n173129\n173392\n173655\n173918\n174181\n174444\n174706\n174969\n175232\n175494\n175757\n176019\n176281\n176544\n176806\n177068\n177330\n177592\n177854\n178116\n178378\n178640\n178902\n179164\n179425\n179687\n179949\n180210\n180472\n180733\n180995\n181256\n181517\n181778\n182040\n182301\n182562\n182823\n183084\n183345\n183606\n183867\n184128\n184389\n184649\n184910\n185171\n185431\n185692\n185953\n186213\n186473\n186734\n186994\n187255\n187515\n187775\n188035\n188296\n188556\n188816\n189076\n189336\n189596\n189856\n190116\n190376\n190635\n190895\n191155\n191415\n191674\n191934\n192194\n192453\n192713\n192972\n193232\n193491\n193750\n194010\n194269\n194528\n194788\n195047\n195306\n195565\n195824\n196084\n196343\n196602\n196861\n197120\n197379\n197638\n197896\n198155\n198414\n198673\n198932\n199190\n199449\n199708\n199966\n200225\n200484\n200742\n201001\n201259\n201518\n201776\n202035\n202293\n202552\n202810\n203068\n203327\n203585\n203843\n204101\n204360\n204618\n204876\n205134\n205392\n205651\n205909\n206167\n206425\n206683\n206941\n207199\n207457\n207715\n207972\n208230\n208488\n208746\n209004\n209262\n209520\n209777\n210035\n210293\n210551\n210808\n211066\n211324\n211581\n211839\n212096\n212354\n212612\n212869\n213127\n213384\n213642\n213899\n214157\n214414\n214671\n214929\n215186\n215444\n215701\n215958\n216216\n216473\n216730\n216988\n217245\n217502\n217759\n218017\n218274\n218531\n218788\n219045\n219303\n219560\n219817\n220074\n220331\n220588\n220845\n221102\n221359\n221616\n221874\n222131\n222388\n222645\n222902\n223158\n223415\n223672\n223929\n224186\n224443\n224700\n224957\n225214\n225471\n225728\n225984\n226241\n226498\n226755\n227012\n227269\n227525\n227782\n228039\n228296\n228552\n228809\n229066\n229323\n229579\n229836\n230093\n230349\n230606\n230863\n231119\n231376\n231633\n231889\n232146\n232403\n232659\n232916\n233173\n233429\n233686\n233942\n234199\n234455\n234712\n234969\n235225\n235482\n235738\n235995\n236251\n236508\n236764\n237021\n237277\n237534\n237790\n238047\n238303\n238560\n238816\n239072\n239329\n239585\n239842\n240098\n240355\n240611\n240867\n241124\n241380\n241637\n241893\n242149\n242406\n242662\n242919\n243175\n243431\n243688\n243944\n244200\n244457\n244713\n244969\n245226\n245482\n245738\n245995\n246251\n246507\n246764\n247020\n247276\n247532\n247789\n248045\n248301\n248557\n248814\n249070\n249326\n249583\n249839\n250095\n250351\n250608\n250864\n251120\n251376\n251633\n251889\n252145\n252401\n252657\n252914\n253170\n253426\n253682\n253938\n254195\n254451\n254707\n254963\n255219\n255476\n255732\n255988\n256244\n256500\n256757\n257013\n257269\n257525\n257781\n258037\n258294\n258550\n258806\n259062\n259318\n259574\n259830\n260087\n260343\n260599\n260855\n261111\n261367\n261623\n261880\n262136\n262392\n262648\n262904\n263160\n263416\n263672\n263929\n264185\n264441\n264697\n264953\n265209\n265465\n265721\n265977\n266234\n266490\n266746\n267002\n267258\n267514\n267770\n268026\n268282\n268538\n268794\n269051\n269307\n269563\n269819\n270075\n270331\n270587\n270843\n271099\n271355\n271611\n271867\n272124\n272380\n272636\n272892\n273148\n273404\n273660\n273916\n274172\n274428\n274684\n274940\n275196\n275452\n275708\n275964\n276221\n276477\n276733\n276989\n277245\n277501\n277757\n278013\n278269\n278525\n278781\n279037\n279293\n279549\n279805\n280061\n280317\n280573\n280829\n281085\n281342\n281598\n281854\n282110\n282366\n282622\n282878\n283134\n283390\n283646\n283902\n284158\n284414\n284670\n284926\n285182\n285438\n285694\n285950\n286206\n286462\n286718\n286974\n287230\n287486\n287742\n287998\n288254\n288510\n288766\n289023\n289279\n289535\n289791\n290047\n290303\n290559\n290815\n291071\n291327\n291583\n291839\n292095\n292351\n292607\n292863\n293119\n293375\n293631\n293887\n294143\n294399\n294655\n294911\n295167\n295423\n295679\n295935\n296191\n296447\n296703\n296959\n297215\n297471\n297727\n297983\n298239\n298495\n298751\n299007\n299263\n299519\n299775\n300031\n300287\n300543\n300799\n301055\n301311\n301567\n301823\n302079\n302335\n302591\n302847\n303103\n303359\n303615\n303871\n304127\n304383\n304639\n304895\n305152\n305408\n305664\n305920\n306176\n306432\n306688\n306944\n307200\n307456\n307712\n307968\n308224\n308480\n308736\n308992\n309248\n309504\n309760\n310016\n310272\n310528\n310784\n311040\n311296\n311552\n311808\n312064\n312320\n312576\n312832\n313088\n313344\n313600\n313856\n314112\n314368\n314624\n314880\n315136\n315392\n315648\n315904\n316160\n316416\n316672\n316928\n317184\n317440\n317696\n317952\n318208\n318464\n318720\n318976\n319232\n319488\n319744\n320000\n320256\n320512\n320768\n321024\n321280\n321536\n321792\n322048\n322304\n322560\n322816\n323072\n323328\n323584\n323840\n324096\n324352\n324608\n324864\n325120\n325376\n325632\n325888\n326144\n326400\n326656\n326912\n327168\n327424\n327680\n327936\n328192\n328448\n328704\n328960\n329216\n329472\n329728\n329984\n330240\n330496\n330752\n331008\n331264\n331520\n331776\n332032\n332288\n332544\n332800\n333056\n333312\n333568\n333824\n334080\n334336\n334592\n334848\n335104\n335360\n335616\n335872\n336128\n336384\n336640\n336896\n337152\n337408\n337664\n337920\n338176\n338432\n338688\n338944\n339200\n339456\n339712\n339968\n340224\n340480\n340736\n340992\n341248\n341504\n341760\n342016\n342272\n342528\n342784\n343040\n343296\n343552\n343808\n344064\n344320\n344576\n344832\n345088\n345344\n345600\n345856\n346112\n346368\n346624\n346880\n347136\n347392\n347648\n347904\n348160\n348416\n348672\n348928\n349184\n349440\n349696\n349952\n350208\n350464\n350720\n350976\n351232\n351488\n351744\n352000\n352256\n352512\n352768\n353024\n353280\n353536\n353792\n354048\n354304\n354560\n354816\n355072\n355328\n355584\n355840\n356096\n356352\n356608\n356864\n357120\n357376\n357632\n357888\n358144\n358400\n358656\n358912\n359168\n359424\n359680\n359936\n360192\n360448\n360704\n360960\n361216\n361472\n361728\n361984\n362240\n362496\n362752\n363008\n363264\n363520\n363776\n364032\n364288\n364544\n364800\n365056\n365312\n365568\n365824\n366080\n366336\n366592\n366848\n367104\n367360\n367616\n367872\n368128\n368384\n368640\n368896\n369152\n369408\n369664\n369920\n370176\n370432\n370688\n370944\n371200\n371456\n371712\n371968\n372224\n372480\n372736\n372992\n373248\n373504\n373760\n374016\n374272\n374528\n374784\n375040\n375296\n375552\n375808\n376064\n376320\n376576\n376832\n377088\n377344\n377600\n377856\n378112\n378368\n378624\n378880\n379136\n379392\n379648\n379904\n380160\n380416\n380672\n380928\n381184\n381440\n381696\n381952\n382208\n382464\n382720\n382976\n383232\n383488\n383744\n384000\n384256\n384512\n384768\n385024\n385280\n385536\n385792\n386048\n386304\n386560\n386816\n387072\n387328\n387584\n387840\n388096\n388352\n388608\n388864\n389120\n389376\n389632\n389888\n390144\n390400\n390656\n390912\n391168\n391424\n391680\n391936\n392192\n392448\n392704\n392960\n393216\n393472\n393728\n393984\n394240\n394496\n394752\n395008\n395264\n395520\n395776\n396032\n396288\n396544\n396800\n397056\n397312\n397568\n397824\n398080\n398336\n398592\n398848\n399104\n399360\n399616\n399872\n400128\n400384\n400640\n400896\n401152\n401408\n401664\n401920\n402176\n402432\n402688\n402944\n403200\n403456\n403712\n403968\n404224\n404480\n404736\n404992\n405248\n405504\n405760\n406016\n406272\n406528\n406784\n407040\n407296\n407552\n407808\n408064\n408320\n408576\n408832\n409088\n409344\n409600\n409856\n410112\n410368\n410624\n410880\n411136\n411392\n411648\n411904\n412160\n412416\n412672\n412928\n413184\n413440\n413696\n413952\n414208\n414464\n414720\n414976\n415232\n415488\n415744\n416000\n416256\n416512\n416768\n417024\n417280\n417536\n417792\n418048\n418304\n418560\n418816\n419072\n419328\n419584\n419840\n420096\n420352\n420608\n420864\n421120\n421376\n421632\n421888\n422144\n422400\n422656\n422912\n423168\n423424\n423680\n423936\n424192\n424448\n424704\n424960\n425216\n425472\n425728\n425984\n426240\n426496\n426752\n427008\n427264\n427520\n427776\n428032\n428288\n428544\n428800\n429056\n429312\n429568\n429824\n430080\n430336\n430592\n430848\n431104\n431360\n431616\n431872\n432128\n432384\n432640\n432896\n433152\n433408\n433664\n433920\n434176\n434432\n434688\n434944\n435200\n435456\n435712\n435968\n436224\n436480\n436736\n436992\n437248\n437504\n437760\n438016\n438272\n438528\n438784\n439040\n439296\n439552\n439808\n440064\n440320\n440576\n440832\n441088\n441344\n441600\n441856\n442112\n442368\n442624\n442880\n443136\n443392\n443648\n443904\n444160\n444416\n444672\n444928\n445184\n445440\n445696\n445952\n446208\n446464\n446720\n446976\n447232\n447488\n447744\n448000\n448256\n448512\n448768\n449024\n449280\n449536\n449792\n450048\n450304\n450560\n450816\n451072\n451328\n451584\n451840\n452096\n452352\n452608\n452864\n453120\n453376\n453632\n453888\n454144\n454400\n454656\n454912\n455168\n455424\n455680\n455936\n456192\n456448\n456704\n456960\n457216\n457472\n457728\n457984\n458240\n458496\n458752\n459008\n459264\n459520\n459776\n460032\n460288\n460544\n460800\n461056\n461312\n461568\n461824\n462080\n462336\n462592\n462848\n463104\n463360\n463616\n463872\n464128\n464384\n464640\n464896\n465152\n465408\n465664\n465920\n466176\n466432\n466688\n466944\n467200\n467456\n467712\n467968\n468224\n468480\n468736\n468992\n469248\n469504\n469760\n470016\n470272\n470528\n470784\n471040\n471296\n471552\n471808\n472064\n472320\n472576\n472832\n473088\n473344\n473600\n473856\n474112\n474368\n474624\n474880\n475136\n475392\n475648\n475904\n476160\n476416\n476672\n476928\n477184\n477440\n477696\n477952\n478208\n478464\n478720\n478976\n479232\n479488\n479744\n480000\n480256\n480512\n480768\n481024\n481280\n481536\n481792\n482048\n482304\n482560\n482816\n483072\n483328\n483584\n483840\n484096\n484352\n484608\n484864\n485120\n485376\n485632\n485888\n486144\n486400\n486656\n486912\n487168\n487424\n487680\n487936\n488192\n488448\n488704\n488960\n489216\n489472\n489728\n489984\n490240\n490496\n490752\n491008\n491264\n491520\n491776\n492032\n492288\n492544\n492800\n493056\n493312\n493568\n493824\n494080\n494336\n494592\n494848\n495104\n495360\n495616\n495872\n496128\n496384\n496640\n496896\n497152\n497408\n497664\n497920\n498176\n498432\n498688\n498944\n499200\n499456\n499712\n499968\n500224\n500480\n500736\n500992\n501248\n501504\n501760\n502016\n502272\n502528\n502784\n503040\n503296\n503552\n503808\n504064\n504320\n504576\n504832\n505088\n505344\n505600\n505856\n506112\n506368\n506624\n506880\n507136\n507392\n507648\n507904\n508160\n508416\n508672\n508928\n509184\n509440\n509696\n509952\n510208\n510464\n510720\n510976\n511232\n511488\n511744\n512000\n512256\n512512\n512768\n513024\n513280\n513536\n513792\n514048\n514304\n514560\n514816\n515072\n515328\n515584\n515840\n516096\n516352\n516608\n516864\n517120\n517376\n517632\n517888\n518144\n518400\n518656\n518912\n519168\n519424\n519680\n519936\n520192\n520448\n520704\n520960\n521216\n521472\n521728\n521984\n522240\n522496\n522752\n523008\n523264\n523520\n523776\n524032\n524288\n')
for name, (digest, body) in FILES.items():
    os.makedirs(os.path.dirname(name) or '.', exist_ok=True)
    with open(name, 'w', encoding='utf-8') as f:
        f.write(body)
    assert hashlib.sha256(open(name, 'rb').read()).hexdigest() == digest, name
print('wrote and checked', ', '.join(FILES))

In [ ]:
import time
import numpy as np
import exact_embed as X

sentences, gold = X.sentences_and_scores()
tok = X.tokenizer()
t0 = time.time()
exact = X.embed(X.Exact(X.load_params()), tok, sentences, 32)
h = X.digest(exact)
print("exact    %s  %s  (%.0f s)" % (h, "MATCH" if h == X.expected() else "DIFFERENT", time.time() - t0))
print("expected %s" % X.expected())

In [ ]:
t0 = time.time()
f32 = X.embed(X.Float32(X.load_params()), tok, sentences, 32)
print("float32  %s  (%.0f s)" % (X.digest(f32), time.time() - t0))

In [ ]:
print("STS-B test, %d pairs, Spearman of cosine against the gold scores:" % len(gold))
print("  exact    %.4f" % X.spearman(X.pair_cosines(exact), gold))
print("  float32  %.4f" % X.spearman(X.pair_cosines(f32), gold))
e = exact.astype(np.float64) / 2.0 ** X.OUT
c = (e * f32).sum(1) / np.linalg.norm(e, axis=1) / np.linalg.norm(f32.astype(np.float64), axis=1)
print("cosine between the float32 and the exact vector of each sentence: min %.6f, mean %.6f" % (c.min(), c.mean()))

In [ ]:
try:
    import jax_runner as J  # sets jax_enable_x64 before any array exists
except ImportError:
    J = None
    print("JAX is not installed here: no GPU run")
gpus = []
if J is not None:
    try:
        gpus = J.jax.devices("gpu")
    except RuntimeError:
        print("no GPU in this runtime: Runtime > Change runtime type > a GPU, then run all cells again")
if gpus:
    v, dev = J.run("gpu", 32)
    h = X.digest(v)
    print("jax int64 on %s  %s  %s" % (dev.device_kind, h, "MATCH" if h == X.expected() else "DIFFERENT"))

In [ ]:
h = X.digest(exact)
print("exact    %s  %s" % (h, "MATCH" if h == X.expected() else "DIFFERENT"))
print("float32  %s  (this machine)" % X.digest(f32))